# NB04 — XGBoost LOOO Cross-Validation (Redesign 28/abr/2026)

Greenfield redesign post-CHG-0007/0008. Preprocessing-agnostic via sklearn `Pipeline`
in caller-provided `model_factory`. Persists 3 tables (NB04 owns):
`looo_predictions`, `looo_metrics`, `looo_model_artifacts`. SQL is authoritative;
downstream NBs (NB05+) read from these tables, NOT from in-memory dicts.

This session: skeleton + **C8** only (crude-only, 168 samples, 42 folds). Remaining
11 configs (C1, C2, C3, C4, C4b, C4c, C6, C7, C2i, Ridge, C_ALL62) deferred to
subsequent sessions once the architecture is validated end-to-end.

**Inputs (read-only):**
- `feature_ml_final` (NB03g) — feature whitelist per config (C45CRUDE / C62ALL)
- `measurements` (NB01) + `diagnostic_ratios` (NB02) — values

**Outputs (this NB owns):**
- `looo_predictions` — per-sample (config × oil × stage) predictions + residuals
- `looo_metrics` — per-config aggregates (overall + per-stage + per-oil_type)
- `looo_model_artifacts` — filesystem refs to pickled models + integrity hash
- Pickles: `models/looo_models/{config_name}/fold_{oil_id}.pkl`

**Decisions binding for this NB:**
- D-W0-CRUDE-42 — drop oils without all 4 stages (C8 primary = 42 oils × 4 = 168)
- D2 (Sessão B) — preprocessing-agnostic `run_looo`; transformations via Pipeline
- D3 (Sessão B) — 3-table schema; last-run-wins idempotence per `config_name`
- M2 (Sessão B) — pickle round-trip sanity (synthetic + first real fold)


## Configuration codes → paper names (crosswalk)

The internal config codes below are **values stored in the database**
(`model_configs.config`, `looo_predictions.config_name`, …) and **model-folder
names** (`models/looo_models/<code>/`), so they are kept verbatim throughout the
pipeline. This legend maps each code to the name used in the paper (Table S2).

| Code | Paper name | Model / feature scope |
|---|---|---|
| `C1` | XGB-all (mixed) | XGBoost, all features, mixed oil types — cohort `C62ALL` (142 features, 44 oils) |
| `C8` | XGB-all (crude) | XGBoost, all features, crude-only — cohort `C45CRUDE` (127 features, 29 oils) |
| `C2` | XGB-ratios | XGBoost, diagnostic ratios only |
| `C3` | XGB-compounds | XGBoost, compounds only |
| `C2i` | XGB-identity | XGBoost, identity-class ratios only |
| `C6` | RF-all | Random Forest, all features |
| `Ridge` | Ridge | RidgeCV linear baseline |
| `C7` | Dummy (crude) | median-prediction baseline, crude scope |
| `C7_mixed` | Dummy (mixed) | median-prediction baseline, mixed scope |
| `C4`, `C4b`, `C4c` | *(not in paper)* | PCA exploratory configs — defined in `model_configs` but **not run** (no predictions/models on disk); excluded from Table S2 |

**Naming caveats.**
- The numerals in `C62ALL` / `C45CRUDE` are **not** counts — the actual counts are 142/127 features and 44/29 oils. Verify by row count, never by the code numerals.
- `W0`–`W3` are the four weathering stages (used as-is in the paper).
- `Sessão …`, `D-…`, `CHG-…`, `§18` are internal process labels (development sessions / decisions / changelog / pre-registration discipline), retained for provenance.


In [15]:
# Imports + setup
import sys
import json
import pickle
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, str(Path.cwd()))
from utils import (
    SEED, PROJECT_ROOT, DB_PATH, MODEL_DIR, FIG_ROOT,
    setup_figure_style, get_conn,
    assert_sterane_canonical_in_db,
    load_ml_dataset,
    run_looo,
    run_looo_optuna,
)

import xgboost as xgb
from utils import ColumnSelector, get_feature_subset_columns
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import RidgeCV
from sklearn.impute import SimpleImputer
import optuna
from typing import Callable, Literal

setup_figure_style()
FIG_DIR = FIG_ROOT / 'nb04'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'xgboost version : {xgb.__version__}  # watchpoint: 2.0.3 → 3.2.0 post Sessão A')
print(f'DB_PATH         : {DB_PATH}')
print(f'MODEL_DIR       : {MODEL_DIR}')
print(f'SEED            : {SEED}')


xgboost version : 3.2.0  # watchpoint: 2.0.3 → 3.2.0 post Sessão A
DB_PATH         : C:\Users\leogr\Documents\Data Science\TCC\data\processed\weathering.db
MODEL_DIR       : C:\Users\leogr\Documents\Data Science\TCC\models\looo_models
SEED            : 42


## Pre-flight invariants

Defense-in-depth check: NB04 is the first consumer of `feature_ml_final` post-CHG-0008.
If any upstream NB drifts (e.g. someone re-runs NB03g without the bottleneck patch),
fail loudly here, not 3 cells later with a mysterious MAE.

Same pattern as the sterane guards from Sessão A — fail high, fail early.


In [16]:
# F-UTILS-1 sterane guard (3 layers: encoding + triggers + ratio category snapshot+pattern)
assert_sterane_canonical_in_db()

# Feature counts post-CHG-0008
with get_conn() as conn:
    n_features_c45 = conn.execute(
        "SELECT COUNT(*) FROM feature_ml_final WHERE config = 'C45CRUDE'"
    ).fetchone()[0]
    n_features_c62 = conn.execute(
        "SELECT COUNT(*) FROM feature_ml_final WHERE config = 'C62ALL'"
    ).fetchone()[0]
    n_protected = conn.execute(
        'SELECT COUNT(*) FROM feature_conditional_value'
    ).fetchone()[0]

assert n_features_c45 == 127, (
    f'feature_ml_final[C45CRUDE] = {n_features_c45}, expected 127 (post-CHG-0008). '
    'Likely upstream drift — check NB03g state.'
)
assert n_features_c62 == 142, (
    f'feature_ml_final[C62ALL] = {n_features_c62}, expected 142 (post-CHG-0008).'
)
assert n_protected == 26, (
    f'feature_conditional_value = {n_protected}, expected 26 (post-CHG-0008).'
)
print(f'✓ pre-flight invariants: 127 C45CRUDE / 142 C62ALL / 26 protected pairs')


✓ assert_sterane_canonical_in_db: L1=3 steranes (ß), L2=2 triggers, L3a=4 snapshot ratios, L3b=4 biomarker×biomarker ratios (all canonical)
✓ pre-flight invariants: 127 C45CRUDE / 142 C62ALL / 26 protected pairs


## D-DATA-SCOPE-NB04 — empirical N per config (post-pivot dropna)

`pandas.pivot_table(dropna=True)` (default) drops index combinations where ALL
aggregated values are NaN. In our data, 14 crudes have 1+ stages where ALL
92 included compound `value_imputed` rows are NULL (Type D from D14 audit
or analytically-empty stages). These stages disappear from the pivot output;
`drop_w0_missing_oils=True` then drops the affected oils entirely.

Empirical reality (this DB state, post-CHG-0007/0008):
- `(C45CRUDE, only_crude=True)` → **29 oils with all 4 stages → 116 samples**

NOT the 42 / 168 implied by D-W0-CRUDE-42, which only catalogued W0 absences
in 2 oils (oil_id 125, 179). The actual scope of partial-stage data is broader
and was not previously surfaced.

This cell discovers and prints the empirical N for each likely config so that
future sessions writing C1, C_ALL62, etc. have the count documented in the
notebook itself, not buried in handoff docs.


In [17]:
# Pre-flight: discover empirical N per config (post-pivot_table dropna)
# Documents D-DATA-SCOPE-NB04 reality check; N is empirical, not asserted.

print('D-DATA-SCOPE-NB04 — empirical N per (feature_set, crude_only):')
print('-' * 70)
with get_conn() as conn:
    for fset, only_c in [('C45CRUDE', True), ('C62ALL', False)]:
        X, y, meta = load_ml_dataset(
            conn,
            include_compounds=True,
            include_ratios=True,
            only_crude=only_c,
        )
        n_oils = meta['oil_id'].nunique()
        stage_counts = meta.groupby('oil_id')['stage_code'].nunique()
        n_complete = int((stage_counts == 4).sum())
        n_partial = int((stage_counts < 4).sum())
        n_samples_complete = n_complete * 4
        print(f'  {fset:>10s} (only_crude={only_c}): '
              f'{n_oils} oils, {n_complete} complete + {n_partial} partial '
              f'→ {n_samples_complete} samples after drop_w0_missing_oils')


D-DATA-SCOPE-NB04 — empirical N per (feature_set, crude_only):
----------------------------------------------------------------------
    C45CRUDE (only_crude=True): 43 oils, 29 complete + 14 partial → 116 samples after drop_w0_missing_oils
      C62ALL (only_crude=False): 60 oils, 44 complete + 16 partial → 176 samples after drop_w0_missing_oils


## xgboost 2→3 pickle round-trip — synthetic (M2 part 1)

Sessão A env repair bumped xgboost 2.0.3 → 3.2.0. Watchpoint: validate pickle
round-trip before trusting any persisted model.

Two-part test (M2):
1. **Synthetic** (this cell) — same hyperparams as C8, fit on tiny synthetic data,
   pickle/load/predict, assert predictions identical. Catches version-format
   incompat at the binary level.
2. **First real fold** (after C8 runs, cell 13) — re-load the first persisted
   artifact, re-predict the held-out oil, assert predictions identical to
   `looo_predictions`. Catches fit-pipeline incompat that synthetic data would
   not expose.


In [18]:
# Synthetic round-trip with C8-identical hyperparams
rng = np.random.default_rng(SEED)
X_syn = pd.DataFrame(
    rng.standard_normal((10, 5)),
    columns=[f'f{i}' for i in range(5)],
)
y_syn = pd.Series(rng.integers(0, 4, size=10), name='stage_int')

# Default hyperparams: regularization-heavy choice for n=168 small dataset.
# NOT a tuned configuration — C8 here is sanity-baseline for end-to-end framework
# validation. C2 will tune via Optuna and produce the headline ML number for Paper B.
C8_HYPERPARAMS = dict(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    tree_method='hist',
    random_state=SEED,
)
syn_model = xgb.XGBRegressor(**C8_HYPERPARAMS)
syn_model.fit(X_syn, y_syn)
pred_pre = syn_model.predict(X_syn)

# Round-trip
buf = pickle.dumps(syn_model)
loaded = pickle.loads(buf)
pred_post = loaded.predict(X_syn)

max_delta = float(np.abs(pred_pre - pred_post).max())
assert np.allclose(pred_pre, pred_post), (
    f'Synthetic pickle round-trip MISMATCH (xgboost format incompat). max |Δ|={max_delta:.2e}'
)
print(f'✓ synthetic round-trip: {len(pred_pre)} preds, max |Δ|={max_delta:.2e}')


✓ synthetic round-trip: 10 preds, max |Δ|=0.00e+00


## NB04 schema setup

DROP + CREATE the 3 tables NB04 owns. Last-run-wins per `config_name` (D3(b)).
FK on `oils(oil_id)` enforced by `PRAGMA foreign_keys = ON` in `get_conn()`.

`run_id` is a UUID12 generated per `run_looo` invocation; preserved on rows for
traceability across re-runs but NOT part of the PK (overwriting metrics on re-run
is the intended idempotence semantic).


In [19]:
NB04_SCHEMA = '''
-- shap_values is owned by NB06 (SHAP redesign pending per CLAUDE.md).
-- NB04 does not manage shap_values lifecycle.

CREATE TABLE IF NOT EXISTS looo_predictions (
    config_name  TEXT NOT NULL,
    oil_id       INTEGER NOT NULL,
    stage_code   TEXT NOT NULL,
    y_true       INTEGER NOT NULL,
    y_pred       REAL NOT NULL,
    residual     REAL NOT NULL,
    abs_error    REAL NOT NULL,
    pm1_correct  INTEGER NOT NULL,
    fold_idx     INTEGER NOT NULL,
    run_id       TEXT NOT NULL,
    created_at   TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (config_name, oil_id, stage_code),
    FOREIGN KEY (oil_id) REFERENCES oils(oil_id)
);

CREATE TABLE IF NOT EXISTS looo_metrics (
    config_name   TEXT NOT NULL,
    metric_name   TEXT NOT NULL,
    metric_scope  TEXT NOT NULL,
    value         REAL NOT NULL,
    n_samples     INTEGER NOT NULL,
    run_id        TEXT NOT NULL,
    created_at    TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (config_name, metric_name, metric_scope)
);

CREATE TABLE IF NOT EXISTS looo_model_artifacts (
    config_name        TEXT NOT NULL,
    fold_idx           INTEGER NOT NULL,
    held_out_oil       INTEGER NOT NULL,
    artifact_path      TEXT NOT NULL,
    n_features         INTEGER NOT NULL,
    n_train_samples    INTEGER NOT NULL,
    train_oil_ids_json TEXT NOT NULL,
    sha256             TEXT,
    run_id             TEXT NOT NULL,
    created_at         TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (config_name, fold_idx),
    FOREIGN KEY (held_out_oil) REFERENCES oils(oil_id)
);

CREATE TABLE IF NOT EXISTS looo_optuna_runs (
    config_name      TEXT    NOT NULL,
    fold_idx         INTEGER NOT NULL,
    held_out_oil     INTEGER NOT NULL,
    run_id           TEXT    NOT NULL,
    n_trials         INTEGER NOT NULL,
    best_value       REAL    NOT NULL,
    best_params_json TEXT    NOT NULL,
    n_inner_splits   INTEGER NOT NULL,
    direction        TEXT    NOT NULL,
    seed             INTEGER,
    created_at       TEXT    DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (config_name, fold_idx, run_id)
);

CREATE TABLE IF NOT EXISTS looo_prediction_intervals (
    config_name    TEXT    NOT NULL,
    oil_id         INTEGER NOT NULL,
    stage_code     TEXT    NOT NULL,
    alpha          REAL    NOT NULL,
    y_pred         REAL    NOT NULL,
    lo             REAL    NOT NULL,
    hi             REAL    NOT NULL,
    in_interval    INTEGER NOT NULL,
    interval_width REAL    NOT NULL,
    n_calibration  INTEGER NOT NULL,
    method         TEXT    NOT NULL,
    run_id         TEXT    NOT NULL,
    created_at     TEXT    DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (config_name, oil_id, stage_code, alpha, run_id)
);

CREATE TABLE IF NOT EXISTS looo_pairwise_tests (
    config_a              TEXT    NOT NULL,
    config_b              TEXT    NOT NULL,
    n_pairs               INTEGER NOT NULL,
    test_name             TEXT    NOT NULL,    -- 'wilcoxon_signed_rank' | 'sign_test'
    statistic             REAL    NOT NULL,
    p_value               REAL    NOT NULL,
    p_corrected           REAL    NOT NULL,
    alpha_family          REAL    NOT NULL,
    n_comparisons         INTEGER NOT NULL,
    effect_size           REAL,                -- Cohen's d_z = mean_diff/sd_diff (Wilcoxon rows)
    mean_diff             REAL,
    median_diff           REAL,
    sd_diff               REAL,
    ci_lo                 REAL,                -- bootstrap 95% CI of median diff
    ci_hi                 REAL,
    sign_pos              INTEGER,
    sign_neg              INTEGER,
    sign_zero             INTEGER,
    min_detectable_r      REAL,                -- d_z MDE at family alpha, n=44, power=0.80
    run_id                TEXT    NOT NULL,
    created_at            TEXT    DEFAULT CURRENT_TIMESTAMP,
    PRIMARY KEY (config_a, config_b, test_name, run_id)
);
'''
with get_conn() as conn:
    conn.executescript(NB04_SCHEMA)

print('✓ schema: looo_predictions, looo_metrics, looo_model_artifacts ensured (IF NOT EXISTS — re-exec safe)')
print('  (post-Spec-D.1: cell[9] re-exec preserves data; shap_values legacy cleanup deferred to NB06 redesign)')


✓ schema: looo_predictions, looo_metrics, looo_model_artifacts ensured (IF NOT EXISTS — re-exec safe)
  (post-Spec-D.1: cell[9] re-exec preserves data; shap_values legacy cleanup deferred to NB06 redesign)


## C8 model factory

C8 = crude-only XGBoost with native NaN handling, no scaling/imputation/log.
Hyperparameters frozen for reproducibility; `model_factory` returns a fresh
estimator per fold (no shared state).

For Pipeline-using configs (C4/C4b/Ridge), the factory composes
`Pipeline([imputer, scaler, ...estimator])` — `run_looo` is preprocessing-agnostic.


In [20]:
def c8_factory():
    """C8: XGBoost regressor, native NaN, no preprocessing pipeline."""
    return xgb.XGBRegressor(**C8_HYPERPARAMS)


# Introspect — auditability win from D2(b) extension (Pipeline-as-config)
demo = c8_factory()
print(f'c8_factory: {type(demo).__name__}')
print(f'  hyperparams: {C8_HYPERPARAMS}')


c8_factory: XGBRegressor
  hyperparams: {'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'tree_method': 'hist', 'random_state': 42}


## Run C8 LOOO

42 folds × 4 stages = 168 samples. 127 features (post-CHG-0008 correlation filter,
8 of which are sterane biomarker ratios destravados via CHG-0007).

`verbose='fold'` prints per-fold MAE for granular feedback. Persists to all 3 tables
+ 42 pickle files in `models/looo_models/C8/`.


In [21]:
result_c8 = run_looo(
    config_name='C8',
    feature_set='C45CRUDE',
    model_factory=c8_factory,
    crude_only=True,
    drop_w0_missing_oils=True,
    persist=True,
    persist_models=True,
    seed=SEED,
    verbose='fold',
)

agg = result_c8['aggregate_metrics']
print()
print('=' * 60)
print(f'C8 SUMMARY (run_id={result_c8["run_id"]})')
print('=' * 60)
print(f'  MAE      : {agg["overall_MAE"]:.3f}')
print(f'  RMSE     : {agg["overall_RMSE"]:.3f}')
print(f'  ±1 stage : {agg["overall_pm1_accuracy"]:.1%}')
print(f'  n_samples: {agg["n_samples"]}  n_folds: {agg["n_folds"]}')


run_looo[C8] run_id=6b493bc7ab33
  feature_set=C45CRUDE crude_only=True drop_w0_missing=True
  drop_w0_missing_oils: removed 14 oils (30 samples). Dropped oil_ids: [124, 136, 174, 176, 179, 183, 187, 194, 213, 226, 227, 232, 237, 239]
  n_oils=29 n_features=127 n_samples=116
  fold  1/29 oil_id= 123 n_test=4 MAE=1.209
  fold  2/29 oil_id= 127 n_test=4 MAE=0.378
  fold  3/29 oil_id= 128 n_test=4 MAE=0.549
  fold  4/29 oil_id= 133 n_test=4 MAE=0.850
  fold  5/29 oil_id= 134 n_test=4 MAE=0.632
  fold  6/29 oil_id= 135 n_test=4 MAE=0.476
  fold  7/29 oil_id= 137 n_test=4 MAE=0.771
  fold  8/29 oil_id= 160 n_test=4 MAE=0.783
  fold  9/29 oil_id= 167 n_test=4 MAE=0.560
  fold 10/29 oil_id= 169 n_test=4 MAE=0.467
  fold 11/29 oil_id= 180 n_test=4 MAE=0.592
  fold 12/29 oil_id= 184 n_test=4 MAE=0.691
  fold 13/29 oil_id= 188 n_test=4 MAE=0.350
  fold 14/29 oil_id= 189 n_test=4 MAE=0.979
  fold 15/29 oil_id= 195 n_test=4 MAE=0.447
  fold 16/29 oil_id= 207 n_test=4 MAE=0.383
  fold 17/29 oil_id=

## First-real-fold pickle round-trip (M2 part 2)

Re-load the first persisted artifact, re-build that oil's feature matrix from the
DB (independent of the in-session result dict), re-predict, and assert
bit-exact match against `looo_predictions`. Catches fit-pipeline format issues
that synthetic data would not expose.


In [22]:
# Pull first artifact metadata
with get_conn() as conn:
    art = pd.read_sql('''
        SELECT config_name, fold_idx, held_out_oil, artifact_path, sha256
        FROM looo_model_artifacts
        WHERE config_name = 'C8'
        ORDER BY fold_idx
        LIMIT 1
    ''', conn)
assert len(art) == 1
held_out = int(art.iloc[0]['held_out_oil'])
artifact_path = PROJECT_ROOT / art.iloc[0]['artifact_path']
print(f'Re-loading: {artifact_path.name} (held_out_oil={held_out})')

with open(artifact_path, 'rb') as f:
    reloaded = pickle.load(f)

# Original predictions for this fold (from looo_predictions)
with get_conn() as conn:
    orig = pd.read_sql('''
        SELECT oil_id, stage_code, y_pred
        FROM looo_predictions
        WHERE config_name = 'C8' AND oil_id = ?
        ORDER BY stage_code
    ''', conn, params=(held_out,))

# Re-build feature matrix for this oil (independent of run_looo's in-memory state)
with get_conn() as conn:
    X_full, _, meta_full = load_ml_dataset(
        conn, include_compounds=True, include_ratios=True, only_crude=True,
    )
    feature_names = pd.read_sql(
        "SELECT feature_name FROM feature_ml_final WHERE config = 'C45CRUDE'",
        conn,
    )['feature_name'].tolist()

available = [f for f in feature_names if f in X_full.columns]
mask = meta_full['oil_id'] == held_out
X_reload = X_full.loc[mask, available]
meta_reload = meta_full.loc[mask].copy()

# Sort by stage_code so it aligns with the SQL ORDER BY
order = meta_reload.sort_values('stage_code').index
X_reload_ordered = X_reload.loc[order]

y_pred_reload = reloaded.predict(X_reload_ordered)
y_pred_orig = orig.sort_values('stage_code')['y_pred'].values

max_delta = float(np.abs(y_pred_reload - y_pred_orig).max())
assert np.allclose(y_pred_reload, y_pred_orig, atol=1e-9), (
    f'First-real-fold round-trip MISMATCH for oil_id={held_out}: max |Δ|={max_delta:.2e}'
)
print(f'✓ first-real-fold round-trip oil_id={held_out}: max |Δ|={max_delta:.2e}')


Re-loading: fold_123.pkl (held_out_oil=123)
✓ first-real-fold round-trip oil_id=123: max |Δ|=0.00e+00


## Validation invariants

Hard asserts on persisted state. If any fail, this run is invalid — do NOT proceed
to C1/C2/.../C_ALL62 in subsequent sessions until fixed.


In [23]:
with get_conn() as conn:
    n_preds = conn.execute(
        "SELECT COUNT(*) FROM looo_predictions WHERE config_name='C8'"
    ).fetchone()[0]
    n_folds_db = conn.execute(
        "SELECT COUNT(DISTINCT oil_id) FROM looo_predictions WHERE config_name='C8'"
    ).fetchone()[0]
    n_artifacts = conn.execute(
        "SELECT COUNT(*) FROM looo_model_artifacts WHERE config_name='C8'"
    ).fetchone()[0]
    mae_overall = conn.execute(
        "SELECT value FROM looo_metrics WHERE config_name='C8' "
        "AND metric_name='MAE' AND metric_scope='all'"
    ).fetchone()[0]
    fk_violations = conn.execute('PRAGMA foreign_key_check(looo_predictions)').fetchall()
    fk_violations += conn.execute('PRAGMA foreign_key_check(looo_model_artifacts)').fetchall()
    n_runids_pred = conn.execute(
        "SELECT COUNT(DISTINCT run_id) FROM looo_predictions WHERE config_name='C8'"
    ).fetchone()[0]
    n_runids_art = conn.execute(
        "SELECT COUNT(DISTINCT run_id) FROM looo_model_artifacts WHERE config_name='C8'"
    ).fetchone()[0]

assert n_preds == 116, f'looo_predictions[C8] = {n_preds}, expected 116 (29 oils × 4 stages, post D-DATA-SCOPE-NB04)'
assert n_folds_db == 29, f'distinct oils = {n_folds_db}, expected 29 (post D-DATA-SCOPE-NB04; 14 crudes dropped beyond D-W0-CRUDE-42 due to partial-stage data)'
assert n_artifacts == 29, f'looo_model_artifacts[C8] = {n_artifacts}, expected 29 (post D-DATA-SCOPE-NB04)'
assert np.isfinite(mae_overall) and mae_overall > 0, f'MAE = {mae_overall} (degenerate or non-finite)'
assert not fk_violations, f'FK violations: {fk_violations}'
assert n_runids_pred == 1, f'multiple run_ids in predictions: {n_runids_pred}'
assert n_runids_art == 1, f'multiple run_ids in artifacts: {n_runids_art}'

# residual = y_true - y_pred consistency
with get_conn() as conn:
    bad_residual = pd.read_sql('''
        SELECT oil_id, stage_code FROM looo_predictions
        WHERE config_name = 'C8' AND ABS(residual - (y_true - y_pred)) > 1e-9
    ''', conn)
assert len(bad_residual) == 0, f'residual inconsistency in {len(bad_residual)} rows'

# abs_error = |residual|
with get_conn() as conn:
    bad_abs = pd.read_sql('''
        SELECT oil_id, stage_code FROM looo_predictions
        WHERE config_name = 'C8' AND ABS(abs_error - ABS(residual)) > 1e-9
    ''', conn)
assert len(bad_abs) == 0, f'abs_error inconsistency in {len(bad_abs)} rows'

# pm1_correct = (abs_error <= 1) consistency
with get_conn() as conn:
    bad_pm1 = pd.read_sql('''
        SELECT oil_id, stage_code FROM looo_predictions
        WHERE config_name = 'C8'
          AND pm1_correct != (CASE WHEN abs_error <= 1 THEN 1 ELSE 0 END)
    ''', conn)
assert len(bad_pm1) == 0, f'pm1_correct inconsistency in {len(bad_pm1)} rows'

print(f'✓ {n_preds} preds / {n_folds_db} folds / {n_artifacts} artifacts (post D-DATA-SCOPE-NB04)')
print(f'✓ residual + abs_error + pm1_correct consistent')
print(f'✓ MAE = {mae_overall:.3f} (finite, positive)')
print(f'✓ FK integrity OK; single run_id across {n_preds} predictions + {n_artifacts} artifacts')


✓ 116 preds / 29 folds / 29 artifacts (post D-DATA-SCOPE-NB04)
✓ residual + abs_error + pm1_correct consistent
✓ MAE = 0.631 (finite, positive)
✓ FK integrity OK; single run_id across 116 predictions + 29 artifacts


## Summary metrics from `looo_metrics`

Snapshot of all aggregate metrics for handoff and downstream NB05+ consumption.
All values read from SQL (authoritative), not from the in-session `result_c8` dict.


In [24]:
with get_conn() as conn:
    metrics = pd.read_sql('''
        SELECT metric_name, metric_scope, value, n_samples, run_id
        FROM looo_metrics
        WHERE config_name = 'C8'
        ORDER BY
            CASE metric_name
                WHEN 'MAE' THEN 1
                WHEN 'RMSE' THEN 2
                WHEN 'pm1_accuracy' THEN 3
                ELSE 9
            END,
            CASE metric_scope WHEN 'all' THEN 0 ELSE 1 END,
            metric_scope
    ''', conn)

print('=' * 70)
print(f'NB04 — C8 final metrics')
print('=' * 70)
print(metrics.drop(columns=['run_id']).to_string(index=False))
print()
print(f"Pickled artifacts: {MODEL_DIR / 'C8'} ({n_artifacts} files)")
print(f"Backup: weathering.db.bak_pre_NB04_<timestamp> (pre-Sessão B)")
print()
print('Next sessions: C1, C2, C3, C4, C4b, C4c, C6, C7, C2i, Ridge, C_ALL62.')
print('Per-config Pipeline strategies in model_factory; run_looo unchanged.')


NB04 — C8 final metrics
 metric_name   metric_scope    value  n_samples
         MAE            all 0.630776        116
         MAE             W0 0.859246         29
         MAE             W1 0.540612         29
         MAE             W2 0.737081         29
         MAE             W3 0.386164         29
         MAE oil_type=crude 0.630776        116
        RMSE            all 0.804805        116
pm1_accuracy            all 0.793103        116
pm1_accuracy             W0 0.689655         29
pm1_accuracy             W1 0.896552         29
pm1_accuracy             W2 0.689655         29
pm1_accuracy             W3 0.896552         29

Pickled artifacts: C:\Users\leogr\Documents\Data Science\TCC\models\looo_models\C8 (29 files)
Backup: weathering.db.bak_pre_NB04_<timestamp> (pre-Sessão B)

Next sessions: C1, C2, C3, C4, C4b, C4c, C6, C7, C2i, Ridge, C_ALL62.
Per-config Pipeline strategies in model_factory; run_looo unchanged.


## Sessão C — C7 baseline + C1 mixed scope (28/apr/2026)

Added in Sessão C to close the prerequisite gate `MAE(C1) < 0.8 × MAE(C7)`
(per HIPOTESES_v1_0) and to produce the paired Wilcoxon comparison C8 vs C7.

**Configs:**
- **C7** — `DummyRegressor(strategy='mean')`, scope identical to C8 (`feature_set='C45CRUDE'` + `crude_only=True`). Baseline statistic. Constant prediction = mean(y_train) = 1.5 (LOOO keeps uniformity {0,1,2,3} per fold).
- **C1** — XGB with the same `C8_HYPERPARAMS`, scope `feature_set='C62ALL'` + `crude_only=False` (mixed oil_types). Contrast with C8 isolates the "effect of including mixed oil_types" under identical hyperparams.

**Idempotence:** each run cell checks `looo_predictions` first; if the config already exists,
skip and read from SQL. To force a re-run, execute manually:
```python
with get_conn() as conn:
    conn.execute("DELETE FROM looo_predictions WHERE config_name='C7'")
    conn.execute("DELETE FROM looo_metrics WHERE config_name='C7'")
    conn.execute("DELETE FROM looo_model_artifacts WHERE config_name='C7'")
```

Original execution: standalone scripts in Sessão C. Cells here are a defensive
re-execution + in-notebook integration of the Sessão C history.


In [25]:
# Sessão C — C7 (DummyRegressor mean baseline, scope C8)
from sklearn.dummy import DummyRegressor


def c7_factory():
    """C7: DummyRegressor strategy='mean' — baseline statistic."""
    return DummyRegressor(strategy='mean')


with get_conn() as conn:
    n_c7_existing = conn.execute(
        "SELECT COUNT(*) FROM looo_predictions WHERE config_name='C7'"
    ).fetchone()[0]

if n_c7_existing > 0:
    print(f'C7 already in looo_predictions ({n_c7_existing} rows). Skipping run; reading from SQL.')
    with get_conn() as conn:
        agg7_overall = conn.execute(
            "SELECT value FROM looo_metrics WHERE config_name='C7' "
            "AND metric_name='MAE' AND metric_scope='all'"
        ).fetchone()[0]
        agg7_pm1 = conn.execute(
            "SELECT value FROM looo_metrics WHERE config_name='C7' "
            "AND metric_name='pm1_accuracy' AND metric_scope='all'"
        ).fetchone()[0]
    print(f'  MAE      : {agg7_overall:.3f}')
    print(f'  ±1 stage : {agg7_pm1:.1%}')
    print(f'  n_samples: {n_c7_existing}')
else:
    result_c7 = run_looo(
        config_name='C7',
        feature_set='C45CRUDE',
        model_factory=c7_factory,
        crude_only=True,
        drop_w0_missing_oils=True,
        persist=True,
        persist_models=True,
        seed=SEED,
        verbose='fold',
    )
    agg7 = result_c7['aggregate_metrics']
    print()
    print('=' * 60)
    print(f'C7 SUMMARY (run_id={result_c7["run_id"]})')
    print('=' * 60)
    print(f'  MAE      : {agg7["overall_MAE"]:.3f}')
    print(f'  RMSE     : {agg7["overall_RMSE"]:.3f}')
    print(f'  ±1 stage : {agg7["overall_pm1_accuracy"]:.1%}')
    print(f'  n_samples: {agg7["n_samples"]}  n_folds: {agg7["n_folds"]}')


run_looo[C7] run_id=9498867830db
  feature_set=C45CRUDE crude_only=True drop_w0_missing=True
  drop_w0_missing_oils: removed 14 oils (30 samples). Dropped oil_ids: [124, 136, 174, 176, 179, 183, 187, 194, 213, 226, 227, 232, 237, 239]
  n_oils=29 n_features=127 n_samples=116
  fold  1/29 oil_id= 123 n_test=4 MAE=1.000
  fold  2/29 oil_id= 127 n_test=4 MAE=1.000
  fold  3/29 oil_id= 128 n_test=4 MAE=1.000
  fold  4/29 oil_id= 133 n_test=4 MAE=1.000
  fold  5/29 oil_id= 134 n_test=4 MAE=1.000
  fold  6/29 oil_id= 135 n_test=4 MAE=1.000
  fold  7/29 oil_id= 137 n_test=4 MAE=1.000
  fold  8/29 oil_id= 160 n_test=4 MAE=1.000
  fold  9/29 oil_id= 167 n_test=4 MAE=1.000
  fold 10/29 oil_id= 169 n_test=4 MAE=1.000
  fold 11/29 oil_id= 180 n_test=4 MAE=1.000
  fold 12/29 oil_id= 184 n_test=4 MAE=1.000
  fold 13/29 oil_id= 188 n_test=4 MAE=1.000
  fold 14/29 oil_id= 189 n_test=4 MAE=1.000
  fold 15/29 oil_id= 195 n_test=4 MAE=1.000
  fold 16/29 oil_id= 207 n_test=4 MAE=1.000
  fold 17/29 oil_id=

In [26]:
# Sessão C — C1 (XGB with C8 hyperparams, scope mixed oil_types)
def c1_factory():
    """C1: same XGB hyperparams as C8, mixed scope (C62ALL + crude_only=False)."""
    return xgb.XGBRegressor(**C8_HYPERPARAMS)


with get_conn() as conn:
    n_c1_existing = conn.execute(
        "SELECT COUNT(*) FROM looo_predictions WHERE config_name='C1'"
    ).fetchone()[0]

if n_c1_existing > 0:
    print(f'C1 already in looo_predictions ({n_c1_existing} rows). Skipping run; reading from SQL.')
    with get_conn() as conn:
        agg1_overall = conn.execute(
            "SELECT value FROM looo_metrics WHERE config_name='C1' "
            "AND metric_name='MAE' AND metric_scope='all'"
        ).fetchone()[0]
        agg1_pm1 = conn.execute(
            "SELECT value FROM looo_metrics WHERE config_name='C1' "
            "AND metric_name='pm1_accuracy' AND metric_scope='all'"
        ).fetchone()[0]
    print(f'  MAE      : {agg1_overall:.3f}')
    print(f'  ±1 stage : {agg1_pm1:.1%}')
    print(f'  n_samples: {n_c1_existing}')
else:
    result_c1 = run_looo(
        config_name='C1',
        feature_set='C62ALL',
        model_factory=c1_factory,
        crude_only=False,
        drop_w0_missing_oils=True,
        persist=True,
        persist_models=True,
        seed=SEED,
        verbose='fold',
    )
    agg1 = result_c1['aggregate_metrics']
    print()
    print('=' * 60)
    print(f'C1 SUMMARY (run_id={result_c1["run_id"]})')
    print('=' * 60)
    print(f'  MAE      : {agg1["overall_MAE"]:.3f}')
    print(f'  RMSE     : {agg1["overall_RMSE"]:.3f}')
    print(f'  ±1 stage : {agg1["overall_pm1_accuracy"]:.1%}')
    print(f'  n_samples: {agg1["n_samples"]}  n_folds: {agg1["n_folds"]}')


run_looo[C1] run_id=4aee320b2817
  feature_set=C62ALL crude_only=False drop_w0_missing=True
  drop_w0_missing_oils: removed 16 oils (34 samples). Dropped oil_ids: [124, 130, 136, 173, 174, 176, 179, 183, 187, 194, 213, 226, 227, 232, 237, 239]
  n_oils=44 n_features=142 n_samples=176
  fold  1/44 oil_id= 122 n_test=4 MAE=0.580
  fold  2/44 oil_id= 123 n_test=4 MAE=1.179
  fold  3/44 oil_id= 127 n_test=4 MAE=0.256
  fold  4/44 oil_id= 128 n_test=4 MAE=0.330
  fold  5/44 oil_id= 133 n_test=4 MAE=1.273
  fold  6/44 oil_id= 134 n_test=4 MAE=0.443
  fold  7/44 oil_id= 135 n_test=4 MAE=0.471
  fold  8/44 oil_id= 137 n_test=4 MAE=1.046
  fold  9/44 oil_id= 160 n_test=4 MAE=0.562
  fold 10/44 oil_id= 162 n_test=4 MAE=0.583
  fold 11/44 oil_id= 163 n_test=4 MAE=0.480
  fold 12/44 oil_id= 164 n_test=4 MAE=0.506
  fold 13/44 oil_id= 165 n_test=4 MAE=0.603
  fold 14/44 oil_id= 166 n_test=4 MAE=0.381
  fold 15/44 oil_id= 167 n_test=4 MAE=0.448
  fold 16/44 oil_id= 169 n_test=4 MAE=0.470
  fold 17/4

In [27]:
# Sessão C — C7_mixed (DummyRegressor mean, scope C1) — strict formal closure
# Pairs with C1 on the same 44 folds for strict Wilcoxon + ratio MAE(C1)/MAE(C7_mixed)
# Math invariance argument: dummy MAE = 1.000 here too (uniform target in LOOO).

with get_conn() as conn:
    n_c7m_existing = conn.execute(
        "SELECT COUNT(*) FROM looo_predictions WHERE config_name='C7_mixed'"
    ).fetchone()[0]

if n_c7m_existing > 0:
    print(f'C7_mixed already in looo_predictions ({n_c7m_existing} rows). Skipping run; reading from SQL.')
    with get_conn() as conn:
        agg7m_overall = conn.execute(
            "SELECT value FROM looo_metrics WHERE config_name='C7_mixed' "
            "AND metric_name='MAE' AND metric_scope='all'"
        ).fetchone()[0]
        agg7m_pm1 = conn.execute(
            "SELECT value FROM looo_metrics WHERE config_name='C7_mixed' "
            "AND metric_name='pm1_accuracy' AND metric_scope='all'"
        ).fetchone()[0]
    print(f'  MAE      : {agg7m_overall:.4f}  (expected 1.000 if target uniform in mixed scope)')
    print(f'  ±1 stage : {agg7m_pm1:.1%}')
    print(f'  n_samples: {n_c7m_existing}')
else:
    result_c7m = run_looo(
        config_name='C7_mixed',
        feature_set='C62ALL',
        model_factory=c7_factory,  # same factory as C7
        crude_only=False,           # MIXED SCOPE — different from C7
        drop_w0_missing_oils=True,
        persist=True,
        persist_models=True,
        seed=SEED,
        verbose='silent',  # 44 folds all equal to 1.000 — silent
    )
    agg7m = result_c7m['aggregate_metrics']
    print()
    print('=' * 60)
    print(f'C7_mixed SUMMARY (run_id={result_c7m["run_id"]})')
    print('=' * 60)
    print(f'  MAE      : {agg7m["overall_MAE"]:.4f}  (expected 1.000)')
    print(f'  RMSE     : {agg7m["overall_RMSE"]:.4f}')
    print(f'  ±1 stage : {agg7m["overall_pm1_accuracy"]:.1%}')
    print(f'  n_samples: {agg7m["n_samples"]}  n_folds: {agg7m["n_folds"]}')

# Strict formal prerequisite gate (C7_mixed paired with C1)
with get_conn() as conn:
    paired = pd.read_sql('''
        SELECT
            (SELECT abs_error FROM looo_predictions WHERE config_name='C1' AND oil_id=lp.oil_id AND stage_code=lp.stage_code) AS c1,
            (SELECT abs_error FROM looo_predictions WHERE config_name='C7_mixed' AND oil_id=lp.oil_id AND stage_code=lp.stage_code) AS c7m
        FROM looo_predictions lp
        WHERE config_name='C1'
    ''', conn)

mae_c1_paired = paired['c1'].mean()
mae_c7m_paired = paired['c7m'].mean()
ratio = mae_c1_paired / mae_c7m_paired

print(f'\n  Strict formal prerequisite gate (C7_mixed paired with C1, n={len(paired)} samples / 44 folds):')
print(f'    MAE(C1) / MAE(C7_mixed) = {mae_c1_paired:.4f} / {mae_c7m_paired:.4f} = {ratio:.4f}')
print(f'    Threshold < 0.80: {"✓ PASSES" if ratio < 0.8 else "✗ FAILS"} (margin {(0.8-ratio)/0.8*100:.1f}%)')
print(f'    → Scenario A confirmed STRICT (HIPOTESES_v1_0 formal prerequisite closed).')



C7_mixed SUMMARY (run_id=6552459f0e63)
  MAE      : 1.0000  (expected 1.000)
  RMSE     : 1.1180
  ±1 stage : 50.0%
  n_samples: 176  n_folds: 44

  Strict formal prerequisite gate (C7_mixed paired com C1, n=176 samples / 44 folds):
    MAE(C1) / MAE(C7_mixed) = 0.4861 / 1.0000 = 0.4861
    Threshold < 0.80: ✓ PASSES (margem 39.2%)
    → Cenário A confirmado STRICT (HIPOTESES_v1_0 prerequisite formal closed).


### Comparative analysis C7 + C1 + C8 (Sessão C)

4 main checks + 2 bonus, reading from SQL (authoritative state). Does not persist
comparative metrics — NB05 will own comparison-metrics persistence.

**Checks:**
1. Wilcoxon paired per-fold MAE C8 vs C7 (29 common folds) — formal significance test
2. Prerequisite gates: `MAE(C1) < 0.8 × MAE(C7)` + `±1 stage ≥ 65%` per HIPOTESES_v1_0
3. C7 prediction constancy verification (DummyRegressor mean = 1.5 always?)
4. Confusion matrix C7 vs C8 — W1 attractor hypothesis test (NB05 hypothesis ii)

**Bonus:**
- (b) Per-stage MAE C7 vs geometric expectation
- C1 vs C8 counter-intuitive comparison (expected C8 ≤ C1; finding inverts the expectation)


In [28]:
# Sessão C — comparative analysis reads from SQL (authoritative)
from scipy.stats import wilcoxon

with get_conn() as conn:
    preds_all = pd.read_sql('''
        SELECT config_name, oil_id, stage_code, y_true, y_pred, residual, abs_error, pm1_correct
        FROM looo_predictions
        WHERE config_name IN ('C7', 'C8', 'C1')
    ''', conn)
print(f'Loaded predictions: {preds_all["config_name"].value_counts().to_dict()}')

# ── (1) Wilcoxon paired per-fold MAE C8 vs C7 ──
print('\n' + '=' * 70)
print('(1) Wilcoxon paired per-fold MAE: C8 vs C7 (29 common folds)')
print('=' * 70)
mae_per_fold = preds_all.groupby(['config_name', 'oil_id'])['abs_error'].mean().unstack('config_name')
paired = mae_per_fold.dropna(subset=['C7', 'C8'])
print(f'Paired folds: {len(paired)}')
print(f'  C8 MAE: mean={paired["C8"].mean():.3f}, median={paired["C8"].median():.3f}, range=[{paired["C8"].min():.3f}, {paired["C8"].max():.3f}]')
print(f'  C7 MAE: mean={paired["C7"].mean():.3f}, median={paired["C7"].median():.3f}, range=[{paired["C7"].min():.3f}, {paired["C7"].max():.3f}]')
stat, p = wilcoxon(paired['C8'], paired['C7'], alternative='less')
print(f'\nWilcoxon (alternative="less", testing C8 < C7):')
print(f'  statistic = {stat:.3f}, p-value = {p:.3e}')
print(f'  Significance @ alpha=0.05: {"✓ YES" if p < 0.05 else "✗ NO"}')

# ── (2) Prerequisite gates ──
print('\n' + '=' * 70)
print('(2) Prerequisite gates per HIPOTESES_v1_0')
print('=' * 70)
mae_c1 = preds_all[preds_all['config_name']=='C1']['abs_error'].mean()
mae_c7 = preds_all[preds_all['config_name']=='C7']['abs_error'].mean()
mae_c8 = preds_all[preds_all['config_name']=='C8']['abs_error'].mean()
pm1_c1 = preds_all[preds_all['config_name']=='C1']['pm1_correct'].mean()
pm1_c8 = preds_all[preds_all['config_name']=='C8']['pm1_correct'].mean()

print(f'  MAE(C1) = {mae_c1:.4f} | MAE(C8) = {mae_c8:.4f} | MAE(C7) = {mae_c7:.4f}')
print(f'\n  (formal) MAE(C1)/MAE(C7) = {mae_c1/mae_c7:.4f} → {"✓ PASS" if mae_c1/mae_c7 < 0.8 else "✗ FAIL"} (threshold < 0.8)')
print(f'  (vibe)   MAE(C8)/MAE(C7) = {mae_c8/mae_c7:.4f} → {"✓ PASS" if mae_c8/mae_c7 < 0.8 else "✗ FAIL"}')
print(f'  ±1 stage C1 = {pm1_c1:.1%} → {"✓ PASS" if pm1_c1 >= 0.65 else "✗ FAIL"} (threshold ≥ 65%)')
print(f'  ±1 stage C8 = {pm1_c8:.1%} → {"✓ PASS" if pm1_c8 >= 0.65 else "✗ FAIL"}')

# Wilcoxon C1 vs C7 paired (29 common folds)
paired_c1_c7 = mae_per_fold.dropna(subset=['C1', 'C7'])
stat_c1c7, p_c1c7 = wilcoxon(paired_c1_c7['C1'], paired_c1_c7['C7'], alternative='less')
print(f'\n  Wilcoxon C1 vs C7 (paired on {len(paired_c1_c7)} common folds):')
print(f'    p-value = {p_c1c7:.3e} (caveat: C7 scope C8; strict formal needs C7 scope C1)')

# ── (3) C7 prediction constancy ──
print('\n' + '=' * 70)
print('(3) C7 prediction constancy verification')
print('=' * 70)
c7 = preds_all[preds_all['config_name']=='C7']
print(f'  C7 unique y_pred: {sorted(c7["y_pred"].unique())}')
print(f'  C7 y_pred mean={c7["y_pred"].mean():.4f}, std={c7["y_pred"].std():.4f}')
print(f'  Expected: 1.5 (mean of {{0,1,2,3}}); LOOO drops 1 oil = 4 stages = uniform → mean stays 1.5 exact')

# ── (4) Confusion matrix C7 vs C8 ──
print('\n' + '=' * 70)
print('(4) Confusion matrix C7 vs C8 — atrator W1 hypothesis test')
print('=' * 70)
for cfg in ['C7', 'C8']:
    sub = preds_all[preds_all['config_name']==cfg].copy()
    sub['y_pred_round'] = np.clip(sub['y_pred'].round().astype(int), 0, 3)
    sub['y_true'] = sub['y_true'].astype(int)
    matrix = pd.crosstab(sub['y_true'], sub['y_pred_round'],
                         rownames=['true'], colnames=['pred'],
                         dropna=False).reindex(index=[0,1,2,3], columns=[0,1,2,3], fill_value=0)
    print(f'\n--- {cfg} ---')
    print(matrix.to_string())
    for s in [0,1,2,3]:
        n_s = (sub['y_true']==s).sum()
        n_correct = ((sub['y_true']==s) & (sub['y_pred_round']==s)).sum()
        print(f'  W{s}: exact={n_correct}/{n_s} ({n_correct/n_s:.0%})')

# ── Bonus (b): Per-stage MAE C7 ──
print('\n' + '=' * 70)
print('Bonus (b): Per-stage MAE C7 vs geometric expectation')
print('=' * 70)
for s in [0,1,2,3]:
    sub = preds_all[(preds_all['config_name']=='C7') & (preds_all['y_true']==s)]
    if len(sub) > 0:
        print(f'  W{s}: C7 MAE = {sub["abs_error"].mean():.3f} (expected geometric |{s} - 1.5| = {abs(s-1.5):.1f})')

# ── Bonus: C1 vs C8 contra-intuitive ──
print('\n' + '=' * 70)
print('Bonus: C1 vs C8 per-stage breakdown (contra-intuitive: C1 wins on every stage)')
print('=' * 70)
print(f'  {"stage":<6} {"C8":>8} {"C1":>8} {"Δ(C8-C1)":>10}')
for s in [0,1,2,3]:
    c8_s = preds_all[(preds_all['config_name']=='C8') & (preds_all['y_true']==s)]['abs_error'].mean()
    c1_s = preds_all[(preds_all['config_name']=='C1') & (preds_all['y_true']==s)]['abs_error'].mean()
    print(f'  W{s:<5} {c8_s:>8.3f} {c1_s:>8.3f} {c8_s-c1_s:>+10.3f}')
print(f'\n  C8 vs C7 per-stage (model worse than dummy in W1/W2 individually):')
print(f'  {"stage":<6} {"C7 (geom)":>10} {"C8":>8} {"C8 wins":>10}')
for s in [0,1,2,3]:
    c8_s = preds_all[(preds_all['config_name']=='C8') & (preds_all['y_true']==s)]['abs_error'].mean()
    c7_geom = abs(s - 1.5)
    print(f'  W{s:<5} {c7_geom:>10.3f} {c8_s:>8.3f} {c7_geom-c8_s:>+10.3f}')


Loaded predictions: {'C1': 176, 'C7': 116, 'C8': 116}

(1) Wilcoxon paired per-fold MAE: C8 vs C7 (29 common folds)
Paired folds: 29
  C8 MAE: mean=0.631, median=0.592, range=[0.348, 1.209]
  C7 MAE: mean=1.000, median=1.000, range=[1.000, 1.000]

Wilcoxon (alternative="less", testing C8 < C7):
  statistic = 7.000, p-value = 3.539e-08
  Significância @ alpha=0.05: ✓ YES

(2) Prerequisite gates per HIPOTESES_v1_0
  MAE(C1) = 0.4861 | MAE(C8) = 0.6308 | MAE(C7) = 1.0000

  (formal) MAE(C1)/MAE(C7) = 0.4861 → ✓ PASS (threshold < 0.8)
  (vibe)   MAE(C8)/MAE(C7) = 0.6308 → ✓ PASS
  ±1 stage C1 = 86.9% → ✓ PASS (threshold ≥ 65%)
  ±1 stage C8 = 79.3% → ✓ PASS

  Wilcoxon C1 vs C7 (paired on 29 common folds):
    p-value = 3.856e-07 (caveat: C7 escopo C8; strict formal needs C7 escopo C1)

(3) C7 prediction constancy verification
  C7 unique y_pred: [1.5]
  C7 y_pred mean=1.5000, std=0.0000
  Expected: 1.5 (mean of {0,1,2,3}); LOOO drops 1 oil = 4 stages = uniform → mean stays 1.5 exact

(4) 

### Sessão C summary — gates + registered decisions

Final print of the Sessão C numbers for handoff. DB backup pre-C7+C1 at
`data/processed/weathering.db.bak_pre_C7C1_<timestamp>`.


In [29]:
# Sessão C — summary print
with get_conn() as conn:
    metrics_all = pd.read_sql('''
        SELECT config_name, metric_name, metric_scope, value, n_samples
        FROM looo_metrics
        WHERE config_name IN ('C7', 'C8', 'C1') AND metric_scope = 'all'
        ORDER BY config_name, CASE metric_name
            WHEN 'MAE' THEN 1 WHEN 'RMSE' THEN 2 WHEN 'pm1_accuracy' THEN 3 ELSE 9 END
    ''', conn)
    n_artifacts = pd.read_sql('''
        SELECT config_name, COUNT(*) AS n
        FROM looo_model_artifacts
        WHERE config_name IN ('C7', 'C8', 'C1')
        GROUP BY config_name
    ''', conn)

print('=' * 70)
print('NB04 SESSÃO C — final state')
print('=' * 70)
print('\nAggregate metrics (overall scope):')
print(metrics_all.to_string(index=False))
print('\nModel artifacts persisted:')
print(n_artifacts.to_string(index=False))
print()
print('Prerequisite gate per HIPOTESES_v1_0: ✓ PASSES (formal + vibe + ±1 stage all green)')
print('Wilcoxon C8 vs C7: p ≈ 3.5e-08 (extremely significant)')
print('Scenario A confirmed. Framework validated.')
print()
print('Emergent findings (Sessão C):')
print('  • C1 (mixed) outperforms C8 (crude-only) across all 4 stages')
print('  • C8 < C7 (dummy) individually in W1, W2 — model only wins at the W0/W3 extremes')
print('  • C8 W1 attractor = boundary-effect + learned attractor (NB05 hypothesis ii partially answered)')
print()
print('Next pending configs (10): C2, C3, C4, C4b, C4c, C6, C2i, Ridge, C_ALL62')


NB04 SESSÃO C — final state

Aggregate metrics (overall scope):
config_name  metric_name metric_scope    value  n_samples
         C1          MAE          all 0.486053        176
         C1         RMSE          all 0.702177        176
         C1 pm1_accuracy          all 0.869318        176
         C7          MAE          all 1.000000        116
         C7         RMSE          all 1.118034        116
         C7 pm1_accuracy          all 0.500000        116
         C8          MAE          all 0.630776        116
         C8         RMSE          all 0.804805        116
         C8 pm1_accuracy          all 0.793103        116

Model artifacts persisted:
config_name  n
         C1 44
         C7 29
         C8 29

Prerequisite gate per HIPOTESES_v1_0: ✓ PASSES (formal + vibe + ±1 stage all green)
Wilcoxon C8 vs C7: p ≈ 3.5e-08 (extremamente significativo)
Cenário A confirmado. Framework validated.

Findings emergentes Sessão C:
  • C1 (mixed) outperforms C8 (crude-only) em tod

## Spec A.3 — NB04_CONFIG_MAP build (Sessão P)

Pipeline-as-config formalization for all 9 Cohort 1 configs. The 4 already-persisted configs (C1, C7, C8, C7_mixed) receive CONFIG_MAP entries for documentation and Spec D (CP) consumption; their stored predictions/artifacts remain canonical (no re-runs in Phase-2A under Q2(b2)). The 5 unrun configs (C2, C2i, C3, C6, Ridge) await Spec C (Optuna integration) before fresh runs.

PCA configs (C4, C4b, C4c) are deferred to a follow-up spec per Path Y decision (Sessão P), pending empirical clarification of upstream feature-source semantics.


In [30]:
# Spec A.3 — non-XGBoost hyperparam defaults (Sessão P)
# These are starting/fixed values. Optuna search spaces for tunable configs
# (Q1: 8 XGB + C6 RF) are added in Spec C; Ridge is fixed-via-RidgeCV-internal-LOO
# per Q1 ratification.

RF_HYPERPARAMS_DEFAULT = {
    'n_estimators': 100,
    'random_state': SEED,
    'n_jobs': -1,
}

RIDGE_HYPERPARAMS_DEFAULT = {
    'alphas': np.logspace(-3, 3, 13),  # Sessão L Block 2 LR canonical grid
    'cv': None,                         # leave-one-out (efficient default for small N)
}

DUMMY_HYPERPARAMS_DEFAULT = {
    'strategy': 'mean',                 # matches existing c7_factory and c7_mixed_factory
}

print(f"RF defaults: {RF_HYPERPARAMS_DEFAULT}")
print(f"Ridge defaults (alphas grid): {len(RIDGE_HYPERPARAMS_DEFAULT['alphas'])} alphas, "
      f"range [{RIDGE_HYPERPARAMS_DEFAULT['alphas'][0]:.0e}, "
      f"{RIDGE_HYPERPARAMS_DEFAULT['alphas'][-1]:.0e}]")
print(f"Dummy defaults: {DUMMY_HYPERPARAMS_DEFAULT}")


RF defaults: {'n_estimators': 100, 'random_state': 42, 'n_jobs': -1}
Ridge defaults (alphas grid): 13 alphas, range [1e-03, 1e+03]
Dummy defaults: {'strategy': 'mean'}


In [31]:
# Spec A.3 — Pipeline-as-config factory builder (Sessão P)

def build_factory(config_name: str, conn):
    """Build a Pipeline-as-config factory for a manifest-declared config.

    Reads model_configs row, resolves feature columns via
    get_feature_subset_columns, and returns a closure that produces a
    fresh sklearn Pipeline when called.

    Cohort 1 only: feature_set ∈ {'all', 'ratios', 'compounds',
    'identity', 'none'}. PCA configs raise ValueError (deferred per
    Path Y, Sessão P).

    Returns
    -------
    Callable[[], Pipeline]
        Factory function with no arguments. Carries metadata as
        attributes: config_name, feature_set_kind, scope, algorithm,
        n_features (empirical post-NB03g count).
    """
    row = conn.execute(
        "SELECT description, feature_set, oil_filter, algorithm "
        "FROM model_configs WHERE config = ?",
        (config_name,)
    ).fetchone()
    if row is None:
        raise KeyError(f"config not found in model_configs: {config_name!r}")
    description, feature_set_kind, oil_filter, algorithm = row

    # Scope derived from oil_filter (NB00 manifest is SSOT)
    scope = 'C45CRUDE' if oil_filter == 'crude_only' else 'C62ALL'

    # Resolve columns at build-time (raises ValueError for 'pca')
    columns = get_feature_subset_columns(conn, feature_set_kind, scope)

    def factory():
        steps = []
        if feature_set_kind != 'none':
            steps.append(('subset', ColumnSelector(columns)))

        if algorithm == 'xgboost':
            steps.append(('xgb', xgb.XGBRegressor(**C8_HYPERPARAMS)))
        elif algorithm == 'random_forest':
            steps.append(('rf', RandomForestRegressor(**RF_HYPERPARAMS_DEFAULT)))
        elif algorithm == 'dummy':
            steps.append(('dummy', DummyRegressor(**DUMMY_HYPERPARAMS_DEFAULT)))
        elif algorithm == 'ridge':
            steps.append(('imputer', SimpleImputer(strategy='median')))
            steps.append(('ridge', RidgeCV(**RIDGE_HYPERPARAMS_DEFAULT)))
        else:
            raise ValueError(f"Unknown algorithm in manifest: {algorithm!r}")

        return Pipeline(steps)

    # Attach metadata for introspection / Spec D consumption
    factory.config_name = config_name
    factory.description = description
    factory.feature_set_kind = feature_set_kind
    factory.scope = scope
    factory.algorithm = algorithm
    factory.n_features = len(columns)
    return factory


In [32]:
# Spec A.3 — NB04_CONFIG_MAP construction (Cohort 1)

COHORT_1_CONFIGS = [
    'C1', 'C2', 'C2i', 'C3', 'C6', 'C7', 'C7_mixed', 'C8', 'Ridge'
]

with get_conn() as conn:
    NB04_CONFIG_MAP = {
        cn: build_factory(cn, conn) for cn in COHORT_1_CONFIGS
    }

# Sanity introspection
print(f"NB04_CONFIG_MAP: {len(NB04_CONFIG_MAP)} configs")
print(f"{'config':<10} {'algorithm':<14} {'scope':<10} {'kind':<11} {'n_features':>11}")
print('-' * 60)
for cn, fac in NB04_CONFIG_MAP.items():
    print(f"{cn:<10} {fac.algorithm:<14} {fac.scope:<10} "
          f"{fac.feature_set_kind:<11} {fac.n_features:>11}")

# Smoke: each factory builds without error
for cn, fac in NB04_CONFIG_MAP.items():
    pipe = fac()
    assert hasattr(pipe, 'fit'), f"{cn} factory output missing .fit"
    assert hasattr(pipe, 'predict'), f"{cn} factory output missing .predict"
print()
print(f"[OK] all {len(NB04_CONFIG_MAP)} factories produce valid Pipelines")


NB04_CONFIG_MAP: 9 configs
config     algorithm      scope      kind         n_features
------------------------------------------------------------
C1         xgboost        C62ALL     all                 142
C2         xgboost        C62ALL     ratios               82
C2i        xgboost        C62ALL     identity             12
C3         xgboost        C62ALL     compounds            60
C6         random_forest  C62ALL     all                 142
C7         dummy          C62ALL     none                  0
C7_mixed   dummy          C62ALL     none                  0
C8         xgboost        C45CRUDE   all                 127
Ridge      ridge          C62ALL     all                 142

[OK] all 9 factories produce valid Pipelines


## Optuna Search Spaces (Q-NEW-C5 β)

Pre-registered hyperparameter search spaces for the 4 tunable Cohort 1 configs (C2, C2i, C3, C6). Each `make_<config>_optuna_factory(conn)` returns a closure compatible with `run_looo_optuna()`'s `model_factory: Callable[[Trial], Pipeline]` contract.

Search spaces calibrated for N≈120 inner-train samples (29 outer folds × ~4 oils × 4 stages); upper bounds capped vs std small-N XGBoost ranges to prevent overfit. **§18 PRE-REG (Sessão Q):** ranges below are methodological commitments; POST-RUN tweaks invalidate pre-registration claim.


In [33]:
# Spec C.2 — Optuna search-space helpers (Sessão Q)

def _xgb_optuna_search(trial: optuna.trial.Trial) -> dict:
    """XGBoost search space, calibrated for N≈120 inner-train samples.

    Upper bounds capped vs std small-N XGBoost ranges to prevent overfit:
        n_estimators 300 (std 500-1000); max_depth 8 (std 10-15);
        reg_alpha/reg_lambda floor 1e-3 (std 1e-8 — numerical instability risk).

    Pre-registered Sessão Q (Spec C.2 v1) — §18 PRE-REG.
    """
    return {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }


def _rf_optuna_search(trial: optuna.trial.Trial) -> dict:
    """RandomForest search space, calibrated for N≈120 inner-train samples.

    Pre-registered Sessão Q (Spec C.2 v1) — §18 PRE-REG.
    """
    return {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5, 1.0]),
    }


In [34]:
# Spec C.2 — Optuna factories (Q-NEW-C5 β closure pattern; Sessão Q)

def make_c2_optuna_factory(conn) -> Callable[[optuna.trial.Trial], Pipeline]:
    """C2 = XGBoost on ratio features (C62ALL scope, kind=ratios; ~82 features)."""
    columns = get_feature_subset_columns(conn, feature_set_kind='ratios', scope='C62ALL')

    def factory(trial: optuna.trial.Trial) -> Pipeline:
        params = _xgb_optuna_search(trial)
        return Pipeline([
            ('subset', ColumnSelector(columns=columns)),
            ('xgb', xgb.XGBRegressor(**params, random_state=SEED, n_jobs=1)),
        ])
    return factory


def make_c2i_optuna_factory(conn) -> Callable[[optuna.trial.Trial], Pipeline]:
    """C2i = XGBoost on identity-ratio features only (C62ALL scope, kind=identity; ~12 features)."""
    columns = get_feature_subset_columns(conn, feature_set_kind='identity', scope='C62ALL')

    def factory(trial: optuna.trial.Trial) -> Pipeline:
        params = _xgb_optuna_search(trial)
        return Pipeline([
            ('subset', ColumnSelector(columns=columns)),
            ('xgb', xgb.XGBRegressor(**params, random_state=SEED, n_jobs=1)),
        ])
    return factory


def make_c3_optuna_factory(conn) -> Callable[[optuna.trial.Trial], Pipeline]:
    """C3 = XGBoost on compound features only (C62ALL scope, kind=compounds; ~60 features)."""
    columns = get_feature_subset_columns(conn, feature_set_kind='compounds', scope='C62ALL')

    def factory(trial: optuna.trial.Trial) -> Pipeline:
        params = _xgb_optuna_search(trial)
        return Pipeline([
            ('subset', ColumnSelector(columns=columns)),
            ('xgb', xgb.XGBRegressor(**params, random_state=SEED, n_jobs=1)),
        ])
    return factory


def make_c6_optuna_factory(conn) -> Callable[[optuna.trial.Trial], Pipeline]:
    """C6 = RandomForest on all features (C62ALL scope, kind=all; ~142 features)."""
    columns = get_feature_subset_columns(conn, feature_set_kind='all', scope='C62ALL')

    def factory(trial: optuna.trial.Trial) -> Pipeline:
        params = _rf_optuna_search(trial)
        return Pipeline([
            ('subset', ColumnSelector(columns=columns)),
            ('rf', RandomForestRegressor(**params, random_state=SEED, n_jobs=1)),
        ])
    return factory


In [35]:
# Spec C.2 — NB04_OPTUNA_FACTORIES registry (Sessão Q)

NB04_OPTUNA_FACTORIES = {
    'C2':  make_c2_optuna_factory,
    'C2i': make_c2i_optuna_factory,
    'C3':  make_c3_optuna_factory,
    'C6':  make_c6_optuna_factory,
}

print(f"NB04_OPTUNA_FACTORIES populated for {len(NB04_OPTUNA_FACTORIES)} configs:")
for cfg_name, fn in NB04_OPTUNA_FACTORIES.items():
    print(f"  {cfg_name}: {fn.__name__}")


NB04_OPTUNA_FACTORIES populated for 4 configs:
  C2: make_c2_optuna_factory
  C2i: make_c2i_optuna_factory
  C3: make_c3_optuna_factory
  C6: make_c6_optuna_factory


## Optuna Driver Helper

The `run_optuna_sweep()` helper iterates over tunable Cohort 1 configs (default: all 4 — C2, C2i, C3, C6) and calls `run_looo_optuna()` per config with metadata pulled via attribute access on `NB04_CONFIG_MAP` factory entries.

Strict Optuna-only scope (Q-NEW-C3.1 ratification): Ridge (fixed via RidgeCV-internal-LOO) and persisted configs (C1/C7/C7_mixed/C8 — retroactive CP sources) are **not** handled by this driver. Ridge runs via `run_looo()` + RidgeCV factory in Spec C.4. Persisted configs are read-only canonical residual sources for Spec D.

**§18 PRE-REG context (Sessão Q):** ranges already pre-registered in C.2 helpers (`_xgb_optuna_search`, `_rf_optuna_search`); driver respects those commitments. POST-RUN range tweaks invalidate pre-registration claim.


In [36]:
# Spec C.3 — run_optuna_sweep() driver (Q-NEW-C5 β; Sessão Q)

def run_optuna_sweep(
    config_names=None,
    *,
    n_trials: int = 50,
    n_inner_splits: int = 3,
    drop_w0_missing_oils: bool = True,
    persist: bool = True,
    persist_models: bool = True,
    seed: int = SEED,
    verbose: Literal['silent', 'fold', 'detailed'] = 'fold',
    show_progress_bar: bool = False,
    db_path=None,
) -> dict:
    """Driver for Optuna HPO sweep across NB04_OPTUNA_FACTORIES.

    Iterates over tunable Cohort 1 configs (default: all 4 — C2, C2i, C3, C6).
    Per-config metadata read via attribute access on NB04_CONFIG_MAP factory
    entries: factory.scope; crude_only derived from scope == 'C45CRUDE' (A.3
    build_factory attaches scope but no separate crude_only attribute).

    Each iteration: builds factory closure via
    NB04_OPTUNA_FACTORIES[config_name](conn), then calls run_looo_optuna()
    with config-specific scope + crude_only + the built factory.

    Non-Optuna configs (Ridge fixed; persisted retroactive C1/C7/C7_mixed/C8) are
    NOT handled here — Ridge gets run_looo() + RidgeCV factory in Spec C.4 5th
    cell; persisted configs are retroactive CP sources (Spec D) and should not
    be auto-rerun.

    config_names:
        None (default) → iterate all NB04_OPTUNA_FACTORIES keys (4 configs)
        list[str]      → iterate only these (must all be in NB04_OPTUNA_FACTORIES)

    Returns:
        dict[config_name -> run_looo_optuna result dict]
        (Per-config result has 6 keys: predictions, fold_metrics,
         aggregate_metrics, model_artifacts, optuna_runs, run_id.)

    Sessão Q (Spec C.3 v2) — Q-NEW-C5 β driver pattern.
    """
    if config_names is None:
        config_names = list(NB04_OPTUNA_FACTORIES.keys())

    unknown = [c for c in config_names if c not in NB04_OPTUNA_FACTORIES]
    if unknown:
        raise ValueError(
            f"Unknown configs (not in NB04_OPTUNA_FACTORIES): {unknown}. "
            f"Available: {list(NB04_OPTUNA_FACTORIES.keys())}"
        )

    db_path_resolved = Path(db_path) if db_path else DB_PATH
    results = {}

    for config_name in config_names:
        cfg_factory_meta = NB04_CONFIG_MAP[config_name]
        feature_set_scope = cfg_factory_meta.scope
        crude_only_cfg = cfg_factory_meta.scope == 'C45CRUDE'

        with get_conn(db_path_resolved) as conn:
            factory = NB04_OPTUNA_FACTORIES[config_name](conn)

        if verbose != 'silent':
            print(f"\n=== run_optuna_sweep[{config_name}] ===")
            print(f"  scope={feature_set_scope} crude_only={crude_only_cfg} "
                  f"n_trials={n_trials} n_inner_splits={n_inner_splits}")

        result = run_looo_optuna(
            config_name=config_name,
            feature_set=feature_set_scope,
            model_factory=factory,
            crude_only=crude_only_cfg,
            drop_w0_missing_oils=drop_w0_missing_oils,
            n_trials=n_trials,
            n_inner_splits=n_inner_splits,
            persist=persist,
            persist_models=persist_models,
            seed=seed,
            verbose=verbose,
            show_progress_bar=show_progress_bar,
            db_path=db_path,
        )
        results[config_name] = result

        if verbose != 'silent':
            agg = result['aggregate_metrics']
            print(f"  → MAE={agg['overall_MAE']:.3f} "
                  f"pm1={agg['overall_pm1_accuracy']:.1%} "
                  f"n={agg['n_samples']} folds={agg['n_folds']}")

    if verbose != 'silent' and len(results) > 1:
        print("\n=== run_optuna_sweep summary ===")
        for cfg, result in results.items():
            agg = result['aggregate_metrics']
            print(f"  {cfg:<5}: MAE={agg['overall_MAE']:.3f} "
                  f"pm1={agg['overall_pm1_accuracy']:.1%}")

    return results


## Spec C.4 — Cohort 1 C-runs (Optuna sweep + Ridge fixed)

This section executes the 4 Optuna-tunable Cohort 1 configs (C2, C2i, C3, C6) via `run_optuna_sweep()` and the Ridge fixed config via `run_looo()` + RidgeCV-internal-LOO alpha selection.

**TIME_BUDGET gate** precedes the full sweep — first-config × few-trials timing → wall-clock projection → halt-or-commit. Persistence: gate non-persistent; full-sweep + Ridge persist via D3(b) last-run-wins.

**§18 PRE-REG (Sessão Q):** hyperparameter ranges pre-registered in C.2 helpers (`_xgb_optuna_search`, `_rf_optuna_search`); driver respects those commitments. POST-RUN range tweaks invalidate pre-registration claim.


In [37]:
# Spec C.4 — TIME_BUDGET gate (Sessão Q)

GATE_N_TRIALS = 5
PRODUCTION_N_TRIALS = 50
N_INNER_SPLITS = 3
TIME_BUDGET_THRESHOLD_MIN = 350  # 5.83 hours; was 120 pre-Q-NEW-C4.6 measurement

print(f"=== TIME_BUDGET gate ===")
print(f"Running C2 with n_trials={GATE_N_TRIALS} for projection (no persist)...")

gate_t0 = time.time()
gate_result = run_optuna_sweep(
    config_names=['C2'],
    n_trials=GATE_N_TRIALS,
    n_inner_splits=N_INNER_SPLITS,
    persist=False,
    persist_models=False,
    verbose='silent',
)
gate_elapsed = time.time() - gate_t0

n_outer_folds = gate_result['C2']['aggregate_metrics']['n_folds']
n_optuna_configs = len(NB04_OPTUNA_FACTORIES)  # 4

# Inner-fit count: trials × outer_folds × inner_splits
gate_fits = GATE_N_TRIALS * n_outer_folds * N_INNER_SPLITS
full_fits = PRODUCTION_N_TRIALS * n_outer_folds * N_INNER_SPLITS * n_optuna_configs

# Add the final-refit count (1 refit per outer fold per config, after Optuna study)
full_refits = n_outer_folds * n_optuna_configs
total_full_fits = full_fits + full_refits

fits_per_sec = gate_fits / gate_elapsed
projected_full_sec = total_full_fits / fits_per_sec
projected_full_min = projected_full_sec / 60

print(f"\nGate:")
print(f"  C2 × {GATE_N_TRIALS} trials × {n_outer_folds} folds × {N_INNER_SPLITS} inner")
print(f"  = {gate_fits:,} inner fits in {gate_elapsed:.1f}s ({fits_per_sec:.1f} fits/s)")
print(f"\nProjection (full sweep, {n_optuna_configs} configs × {PRODUCTION_N_TRIALS} trials):")
print(f"  Inner fits: {full_fits:,}")
print(f"  Final refits: {full_refits:,}")
print(f"  Total fits: {total_full_fits:,}")
print(f"  Estimated wall-clock: {projected_full_min:.1f} min ({projected_full_min/60:.2f} h)")
print(f"  Threshold: {TIME_BUDGET_THRESHOLD_MIN} min")

if projected_full_min > TIME_BUDGET_THRESHOLD_MIN:
    raise RuntimeError(
        f"Projected {projected_full_min:.0f} min exceeds threshold {TIME_BUDGET_THRESHOLD_MIN} min. "
        f"To proceed: (1) edit TIME_BUDGET_THRESHOLD_MIN above; (2) reduce PRODUCTION_N_TRIALS; "
        f"(3) accept and re-run cell to re-measure (TPE sampler timing varies)."
    )

print(f"\n→ TIME_BUDGET gate PASSED. Safe to proceed to per-config cells.")


=== TIME_BUDGET gate ===
Running C2 with n_trials=5 for projection (no persist)...


[I 2026-05-16 08:29:37,882] A new study created in memory with name: no-name-ff71c808-8d59-46d8-bf58-2212b449a7b9
[I 2026-05-16 08:29:39,558] Trial 0 finished with value: 0.5904162327448527 and parameters: {'n_estimators': 144, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 0.5904162327448527.
[I 2026-05-16 08:29:41,435] Trial 1 finished with value: 0.5774741570154825 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.010725209743171996, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 1 with value: 0.5774741570154825.
[I 2026-05-16 08:29:42,489] Trial 2 finished with value: 0.5543841322263082 and parameters: {'n_estimators': 126, 'max_d


Gate:
  C2 × 5 trials × 44 folds × 3 inner
  = 660 inner fits in 418.8s (1.6 fits/s)

Projection (full sweep, 4 configs × 50 trials):
  Inner fits: 26,400
  Final refits: 176
  Total fits: 26,576
  Estimated wall-clock: 281.1 min (4.68 h)
  Threshold: 350 min

→ TIME_BUDGET gate PASSED. Safe to proceed to per-config cells.


In [38]:
# Spec C.4 — C2 full Optuna run (Sessão Q)
print(f"=== Running C2 Optuna sweep (n_trials={PRODUCTION_N_TRIALS}) ===")
c2_results = run_optuna_sweep(
    config_names=['C2'],
    n_trials=PRODUCTION_N_TRIALS,
    n_inner_splits=N_INNER_SPLITS,
    persist=True,
    persist_models=True,
    verbose='fold',
)


[I 2026-05-16 08:36:36,665] A new study created in memory with name: no-name-1bae0c0f-3904-4ac8-8ab5-3d445a557805


=== Running C2 Optuna sweep (n_trials=50) ===

=== run_optuna_sweep[C2] ===
  scope=C62ALL crude_only=False n_trials=50 n_inner_splits=3
run_looo_optuna[C2] run_id=d325331e4e72
  feature_set=C62ALL crude_only=False drop_w0_missing=True n_trials=50 n_inner_splits=3
  drop_w0_missing_oils: removed 16 oils (34 samples). Dropped oil_ids: [124, 130, 136, 173, 174, 176, 179, 183, 187, 194, 213, 226, 227, 232, 237, 239]
  n_oils=44 n_features=142 n_samples=176


[I 2026-05-16 08:36:38,310] Trial 0 finished with value: 0.5904162327448527 and parameters: {'n_estimators': 144, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 0.5904162327448527.
[I 2026-05-16 08:36:40,593] Trial 1 finished with value: 0.5774741570154825 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.010725209743171996, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 1 with value: 0.5774741570154825.
[I 2026-05-16 08:36:41,976] Trial 2 finished with value: 0.5543841322263082 and parameters: {'n_estimators': 126, 'max_depth': 6, 'learning_rate': 0.04345454109729477, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.84474115788

  fold  1/44 oil_id= 122 n_test=4 MAE=0.664 best=0.533 n_trials=50


[I 2026-05-16 08:37:31,233] Trial 0 finished with value: 0.5623966654141744 and parameters: {'n_estimators': 144, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 0.5623966654141744.
[I 2026-05-16 08:37:33,447] Trial 1 finished with value: 0.5607136090596517 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.010725209743171996, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 1 with value: 0.5607136090596517.
[I 2026-05-16 08:37:34,701] Trial 2 finished with value: 0.5344456632932028 and parameters: {'n_estimators': 126, 'max_depth': 6, 'learning_rate': 0.04345454109729477, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.84474115788

...

  fold 42/44 oil_id= 236 n_test=4 MAE=0.522 best=0.498 n_trials=50


[I 2026-05-16 09:11:11,751] Trial 0 finished with value: 0.5591581463813782 and parameters: {'n_estimators': 144, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 0.5591581463813782.
[I 2026-05-16 09:11:13,416] Trial 1 finished with value: 0.5914423664410909 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.010725209743171996, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 0 with value: 0.5591581463813782.
[I 2026-05-16 09:11:14,367] Trial 2 finished with value: 0.5413093368212382 and parameters: {'n_estimators': 126, 'max_depth': 6, 'learning_rate': 0.04345454109729477, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.84474115788

  fold 43/44 oil_id= 238 n_test=4 MAE=0.572 best=0.506 n_trials=50


[I 2026-05-16 09:11:46,549] Trial 0 finished with value: 0.5561882853507996 and parameters: {'n_estimators': 144, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 0.5561882853507996.
[I 2026-05-16 09:11:48,199] Trial 1 finished with value: 0.5830237865447998 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.010725209743171996, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 0 with value: 0.5561882853507996.
[I 2026-05-16 09:11:49,120] Trial 2 finished with value: 0.5310428241888682 and parameters: {'n_estimators': 126, 'max_depth': 6, 'learning_rate': 0.04345454109729477, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.84474115788

  fold 44/44 oil_id= 240 n_test=4 MAE=0.499 best=0.508 n_trials=50
  → MAE=0.522 RMSE=0.693 pm1=83.5% n=176 (Optuna: 50 trials × 3-fold inner CV × 44 folds)
  → MAE=0.522 pm1=83.5% n=176 folds=44


In [39]:
# Spec C.4 — C2i full Optuna run (Sessão Q)
print(f"=== Running C2i Optuna sweep (n_trials={PRODUCTION_N_TRIALS}) ===")
c2i_results = run_optuna_sweep(
    config_names=['C2i'],
    n_trials=PRODUCTION_N_TRIALS,
    n_inner_splits=N_INNER_SPLITS,
    persist=True,
    persist_models=True,
    verbose='fold',
)


=== Running C2i Optuna sweep (n_trials=50) ===

=== run_optuna_sweep[C2i] ===
  scope=C62ALL crude_only=False n_trials=50 n_inner_splits=3
run_looo_optuna[C2i] run_id=59d4d0be1c9d
  feature_set=C62ALL crude_only=False drop_w0_missing=True n_trials=50 n_inner_splits=3


[I 2026-05-16 09:12:36,673] A new study created in memory with name: no-name-1a129e6c-1cdb-42d4-9166-90eef1f6d639
[I 2026-05-16 09:12:36,867] Trial 0 finished with value: 1.030073602994283 and parameters: {'n_estimators': 144, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 1.030073602994283.


  drop_w0_missing_oils: removed 16 oils (34 samples). Dropped oil_ids: [124, 130, 136, 173, 174, 176, 179, 183, 187, 194, 213, 226, 227, 232, 237, 239]
  n_oils=44 n_features=142 n_samples=176


[I 2026-05-16 09:12:37,136] Trial 1 finished with value: 1.0245933532714844 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.010725209743171996, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 1 with value: 1.0245933532714844.
[I 2026-05-16 09:12:37,289] Trial 2 finished with value: 1.046351949373881 and parameters: {'n_estimators': 126, 'max_depth': 6, 'learning_rate': 0.04345454109729477, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.8447411578889518, 'min_child_weight': 2, 'reg_alpha': 0.01474275315991467, 'reg_lambda': 0.029204338471814112}. Best is trial 1 with value: 1.0245933532714844.
[I 2026-05-16 09:12:37,589] Trial 3 finished with value: 1.0071847240130107 and parameters: {'n_estimators': 164, 'max_depth': 7, 'learning_rate': 0.019721610970574007, 'subsample': 0.8056937753654446, 'colsample_bytree': 0.8369658275

  fold  1/44 oil_id= 122 n_test=4 MAE=0.946 best=0.989 n_trials=50


...

  fold 42/44 oil_id= 236 n_test=4 MAE=1.001 best=0.994 n_trials=50


[I 2026-05-16 09:18:41,846] Trial 1 finished with value: 1.0307227770487468 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.010725209743171996, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 0 with value: 1.0192581216494243.
[I 2026-05-16 09:18:42,002] Trial 2 finished with value: 1.0362608035405476 and parameters: {'n_estimators': 126, 'max_depth': 6, 'learning_rate': 0.04345454109729477, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.8447411578889518, 'min_child_weight': 2, 'reg_alpha': 0.01474275315991467, 'reg_lambda': 0.029204338471814112}. Best is trial 0 with value: 1.0192581216494243.
[I 2026-05-16 09:18:42,307] Trial 3 finished with value: 1.0134858687718709 and parameters: {'n_estimators': 164, 'max_depth': 7, 'learning_rate': 0.019721610970574007, 'subsample': 0.8056937753654446, 'colsample_bytree': 0.836965827

  fold 43/44 oil_id= 238 n_test=4 MAE=1.062 best=0.998 n_trials=50


[I 2026-05-16 09:18:48,468] Trial 1 finished with value: 1.030611475308736 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.010725209743171996, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 0 with value: 1.016076425711314.
[I 2026-05-16 09:18:48,635] Trial 2 finished with value: 1.0285976926485698 and parameters: {'n_estimators': 126, 'max_depth': 6, 'learning_rate': 0.04345454109729477, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.8447411578889518, 'min_child_weight': 2, 'reg_alpha': 0.01474275315991467, 'reg_lambda': 0.029204338471814112}. Best is trial 0 with value: 1.016076425711314.
[I 2026-05-16 09:18:48,914] Trial 3 finished with value: 1.0054255723953247 and parameters: {'n_estimators': 164, 'max_depth': 7, 'learning_rate': 0.019721610970574007, 'subsample': 0.8056937753654446, 'colsample_bytree': 0.836965827544

  fold 44/44 oil_id= 240 n_test=4 MAE=0.962 best=0.994 n_trials=50
  → MAE=0.998 RMSE=1.125 pm1=47.2% n=176 (Optuna: 50 trials × 3-fold inner CV × 44 folds)
  → MAE=0.998 pm1=47.2% n=176 folds=44


In [40]:
# Spec C.4 — C3 full Optuna run (Sessão Q)
print(f"=== Running C3 Optuna sweep (n_trials={PRODUCTION_N_TRIALS}) ===")
c3_results = run_optuna_sweep(
    config_names=['C3'],
    n_trials=PRODUCTION_N_TRIALS,
    n_inner_splits=N_INNER_SPLITS,
    persist=True,
    persist_models=True,
    verbose='fold',
)


[I 2026-05-16 09:18:58,202] A new study created in memory with name: no-name-35b3b2db-8a44-46cc-87b2-7eac961cafb6


=== Running C3 Optuna sweep (n_trials=50) ===

=== run_optuna_sweep[C3] ===
  scope=C62ALL crude_only=False n_trials=50 n_inner_splits=3
run_looo_optuna[C3] run_id=8a285fa512ec
  feature_set=C62ALL crude_only=False drop_w0_missing=True n_trials=50 n_inner_splits=3
  drop_w0_missing_oils: removed 16 oils (34 samples). Dropped oil_ids: [124, 130, 136, 173, 174, 176, 179, 183, 187, 194, 213, 226, 227, 232, 237, 239]
  n_oils=44 n_features=142 n_samples=176


[I 2026-05-16 09:18:58,860] Trial 0 finished with value: 0.5472238262494405 and parameters: {'n_estimators': 144, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 0.5472238262494405.
[I 2026-05-16 09:18:59,956] Trial 1 finished with value: 0.5244204699993134 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.010725209743171996, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 1 with value: 0.5244204699993134.
[I 2026-05-16 09:19:00,531] Trial 2 finished with value: 0.4932486017545064 and parameters: {'n_estimators': 126, 'max_depth': 6, 'learning_rate': 0.04345454109729477, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.84474115788

  fold  1/44 oil_id= 122 n_test=4 MAE=0.546 best=0.485 n_trials=50


[I 2026-05-16 09:19:29,124] Trial 0 finished with value: 0.5202293495337168 and parameters: {'n_estimators': 144, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 0.5202293495337168.
[I 2026-05-16 09:19:30,204] Trial 1 finished with value: 0.5044932762781779 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.010725209743171996, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 1 with value: 0.5044932762781779.
[I 2026-05-16 09:19:30,757] Trial 2 finished with value: 0.4789183537165324 and parameters: {'n_estimators': 126, 'max_depth': 6, 'learning_rate': 0.04345454109729477, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.84474115788

...

  fold 42/44 oil_id= 236 n_test=4 MAE=0.536 best=0.474 n_trials=50


[I 2026-05-16 09:38:22,214] Trial 0 finished with value: 0.5510333478450775 and parameters: {'n_estimators': 144, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 0.5510333478450775.
[I 2026-05-16 09:38:23,309] Trial 1 finished with value: 0.54727370540301 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.010725209743171996, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 1 with value: 0.54727370540301.
[I 2026-05-16 09:38:23,909] Trial 2 finished with value: 0.4971144497394562 and parameters: {'n_estimators': 126, 'max_depth': 6, 'learning_rate': 0.04345454109729477, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.844741157888951

  fold 43/44 oil_id= 238 n_test=4 MAE=0.434 best=0.473 n_trials=50


[I 2026-05-16 09:38:49,018] Trial 0 finished with value: 0.5734575688838959 and parameters: {'n_estimators': 144, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'min_child_weight': 2, 'reg_alpha': 0.0017073967431528124, 'reg_lambda': 2.9154431891537547}. Best is trial 0 with value: 0.5734575688838959.
[I 2026-05-16 09:38:50,100] Trial 1 finished with value: 0.552664707104365 and parameters: {'n_estimators': 200, 'max_depth': 7, 'learning_rate': 0.010725209743171996, 'subsample': 0.9879639408647978, 'colsample_bytree': 0.9329770563201687, 'min_child_weight': 3, 'reg_alpha': 0.005337032762603957, 'reg_lambda': 0.00541524411940254}. Best is trial 1 with value: 0.552664707104365.
[I 2026-05-16 09:38:50,685] Trial 2 finished with value: 0.5108886162439982 and parameters: {'n_estimators': 126, 'max_depth': 6, 'learning_rate': 0.04345454109729477, 'subsample': 0.7164916560792167, 'colsample_bytree': 0.8447411578889

  fold 44/44 oil_id= 240 n_test=4 MAE=0.245 best=0.479 n_trials=50
  → MAE=0.495 RMSE=0.702 pm1=88.6% n=176 (Optuna: 50 trials × 3-fold inner CV × 44 folds)
  → MAE=0.495 pm1=88.6% n=176 folds=44


In [41]:
# Spec C.4 — C6 full Optuna run (Sessão Q)
print(f"=== Running C6 Optuna sweep (n_trials={PRODUCTION_N_TRIALS}) ===")
c6_results = run_optuna_sweep(
    config_names=['C6'],
    n_trials=PRODUCTION_N_TRIALS,
    n_inner_splits=N_INNER_SPLITS,
    persist=True,
    persist_models=True,
    verbose='fold',
)


[I 2026-05-16 09:39:14,336] A new study created in memory with name: no-name-e13dc4fb-0d32-4576-b65f-1c6c02a3c1ba


=== Running C6 Optuna sweep (n_trials=50) ===

=== run_optuna_sweep[C6] ===
  scope=C62ALL crude_only=False n_trials=50 n_inner_splits=3
run_looo_optuna[C6] run_id=88500677a9ce
  feature_set=C62ALL crude_only=False drop_w0_missing=True n_trials=50 n_inner_splits=3
  drop_w0_missing_oils: removed 16 oils (34 samples). Dropped oil_ids: [124, 130, 136, 173, 174, 176, 179, 183, 187, 194, 213, 226, 227, 232, 237, 239]
  n_oils=44 n_features=142 n_samples=176


[I 2026-05-16 09:39:16,623] Trial 0 finished with value: 0.5324934526403126 and parameters: {'n_estimators': 250, 'max_depth': 15, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 1.0}. Best is trial 0 with value: 0.5324934526403126.
[I 2026-05-16 09:39:17,495] Trial 1 finished with value: 0.6484878622785245 and parameters: {'n_estimators': 341, 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5324934526403126.
[I 2026-05-16 09:39:18,155] Trial 2 finished with value: 0.5929251514759857 and parameters: {'n_estimators': 222, 'max_depth': 9, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5324934526403126.
[I 2026-05-16 09:39:19,761] Trial 3 finished with value: 0.5448598277126177 and parameters: {'n_estimators': 282, 'max_depth': 13, 'min_samples_split': 5, 'min_samples_leaf': 6, 'max_features': 0.5}. Best is trial 0 with value: 0.5324934526403126.


  fold  1/44 oil_id= 122 n_test=4 MAE=0.523 best=0.511 n_trials=50


[I 2026-05-16 09:41:40,361] Trial 0 finished with value: 0.5122995040627096 and parameters: {'n_estimators': 250, 'max_depth': 15, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 1.0}. Best is trial 0 with value: 0.5122995040627096.
[I 2026-05-16 09:41:41,252] Trial 1 finished with value: 0.6382986708815348 and parameters: {'n_estimators': 341, 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5122995040627096.
[I 2026-05-16 09:41:41,970] Trial 2 finished with value: 0.5815217475252973 and parameters: {'n_estimators': 222, 'max_depth': 9, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5122995040627096.
[I 2026-05-16 09:41:43,606] Trial 3 finished with value: 0.5293811235059737 and parameters: {'n_estimators': 282, 'max_depth': 13, 'min_samples_split': 5, 'min_samples_leaf': 6, 'max_features': 0.5}. Best is trial 0 with value: 0.5122995040627096.


...

  fold 42/44 oil_id= 236 n_test=4 MAE=0.529 best=0.506 n_trials=50


[I 2026-05-16 11:23:46,174] Trial 0 finished with value: 0.5277459746377935 and parameters: {'n_estimators': 250, 'max_depth': 15, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 1.0}. Best is trial 0 with value: 0.5277459746377935.
[I 2026-05-16 11:23:47,033] Trial 1 finished with value: 0.6441604311666757 and parameters: {'n_estimators': 341, 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5277459746377935.
[I 2026-05-16 11:23:47,657] Trial 2 finished with value: 0.5903312182502193 and parameters: {'n_estimators': 222, 'max_depth': 9, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5277459746377935.
[I 2026-05-16 11:23:49,255] Trial 3 finished with value: 0.5385480050170272 and parameters: {'n_estimators': 282, 'max_depth': 13, 'min_samples_split': 5, 'min_samples_leaf': 6, 'max_features': 0.5}. Best is trial 0 with value: 0.5277459746377935.


  fold 43/44 oil_id= 238 n_test=4 MAE=0.483 best=0.505 n_trials=50


[I 2026-05-16 11:26:37,528] Trial 0 finished with value: 0.5358984248507072 and parameters: {'n_estimators': 250, 'max_depth': 15, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 1.0}. Best is trial 0 with value: 0.5358984248507072.
[I 2026-05-16 11:26:38,373] Trial 1 finished with value: 0.6406354366371336 and parameters: {'n_estimators': 341, 'max_depth': 12, 'min_samples_split': 2, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5358984248507072.
[I 2026-05-16 11:26:39,006] Trial 2 finished with value: 0.5920058743860716 and parameters: {'n_estimators': 222, 'max_depth': 9, 'min_samples_split': 10, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5358984248507072.
[I 2026-05-16 11:26:40,697] Trial 3 finished with value: 0.541265361487991 and parameters: {'n_estimators': 282, 'max_depth': 13, 'min_samples_split': 5, 'min_samples_leaf': 6, 'max_features': 0.5}. Best is trial 0 with value: 0.5358984248507072.
[

  fold 44/44 oil_id= 240 n_test=4 MAE=0.394 best=0.514 n_trials=50
  → MAE=0.496 RMSE=0.657 pm1=90.9% n=176 (Optuna: 50 trials × 3-fold inner CV × 44 folds)
  → MAE=0.496 pm1=90.9% n=176 folds=44


In [42]:
# Spec C.4 — Ridge fixed (no Optuna; RidgeCV-internal-LOO alpha selection)
ridge_meta = NB04_CONFIG_MAP['Ridge']  # callable factory + attached attrs
ridge_factory = ridge_meta              # same object; semantic alias for clarity

print(f"=== Running Ridge (RidgeCV alphas grid via internal LOO) ===")
print(f"  scope={ridge_meta.scope} algorithm={ridge_meta.algorithm} "
      f"n_features={ridge_meta.n_features}")

ridge_result = run_looo(
    config_name='Ridge',
    feature_set=ridge_meta.scope,
    model_factory=ridge_factory,
    crude_only=ridge_meta.scope == 'C45CRUDE',
    drop_w0_missing_oils=True,
    persist=True,
    persist_models=True,
    seed=SEED,
    verbose='fold',
)

agg = ridge_result['aggregate_metrics']
print(f"\n→ Ridge: MAE={agg['overall_MAE']:.3f} "
      f"pm1={agg['overall_pm1_accuracy']:.1%} "
      f"n={agg['n_samples']} folds={agg['n_folds']}")


=== Running Ridge (RidgeCV alphas grid via internal LOO) ===
  scope=C62ALL algorithm=ridge n_features=142
run_looo[Ridge] run_id=3e789f5b5997
  feature_set=C62ALL crude_only=False drop_w0_missing=True
  drop_w0_missing_oils: removed 16 oils (34 samples). Dropped oil_ids: [124, 130, 136, 173, 174, 176, 179, 183, 187, 194, 213, 226, 227, 232, 237, 239]
  n_oils=44 n_features=142 n_samples=176
  fold  1/44 oil_id= 122 n_test=4 MAE=0.693
  fold  2/44 oil_id= 123 n_test=4 MAE=1.180
  fold  3/44 oil_id= 127 n_test=4 MAE=0.993
  fold  4/44 oil_id= 128 n_test=4 MAE=0.480
  fold  5/44 oil_id= 133 n_test=4 MAE=1.024
  fold  6/44 oil_id= 134 n_test=4 MAE=0.621
  fold  7/44 oil_id= 135 n_test=4 MAE=1.105
  fold  8/44 oil_id= 137 n_test=4 MAE=0.619
  fold  9/44 oil_id= 160 n_test=4 MAE=3.608
  fold 10/44 oil_id= 162 n_test=4 MAE=0.519
  fold 11/44 oil_id= 163 n_test=4 MAE=0.947
  fold 12/44 oil_id= 164 n_test=4 MAE=0.672
  fold 13/44 oil_id= 165 n_test=4 MAE=0.629
  fold 14/44 oil_id= 166 n_test=4

## Spec D — Conformal Prediction Intervals (LOO conformal, retroactive Cohort 1)

For each Cohort 1 config, compute leave-one-out conformal prediction intervals at α=0.05 and α=0.10. Method: symmetric `ŷ ± q_(1-α)(|residual|)` using residuals from all OTHER oils as calibration (per Q-NEW-D.2 pooled-all-stages lean).

Coverage guarantee: ≥ (1-α) under residual exchangeability. Method distinct from Barber 2021 jackknife+ (which requires N×N prediction matrix; not retroactively available). Documented as `loo_conformal_symmetric` in `looo_prediction_intervals.method` column.

**Diagnostic:** empirical coverage vs nominal target, broken down by config × α × stage × oil_type — Paper B coverage tables source.


In [43]:
# Spec D — compute LOO conformal intervals for all Cohort 1 configs
import sys
if 'utils' in sys.modules:
    del sys.modules['utils']
from utils import compute_loo_conformal_intervals

ALPHAS = (0.05, 0.10)
COHORT_1_FOR_CP = list(NB04_CONFIG_MAP.keys())  # 9 configs

cp_results = {}
for config_name in COHORT_1_FOR_CP:
    print(f"\n=== Computing LOO conformal intervals for {config_name} ===")
    df_intervals = compute_loo_conformal_intervals(
        config_name=config_name,
        alphas=ALPHAS,
        persist=True,
        verbose='fold',
    )
    cp_results[config_name] = df_intervals

print(f"\n=== Spec D complete: {len(cp_results)} configs × {len(ALPHAS)} alphas persisted ===")


=== Computing LOO conformal intervals for C1 ===
  α=0.05 target=95% empirical=95.5% mean_width=3.050 n=176
  α=0.10 target=90% empirical=90.3% mean_width=2.430 n=176

=== Computing LOO conformal intervals for C2 ===
  α=0.05 target=95% empirical=95.5% mean_width=3.078 n=176
  α=0.10 target=90% empirical=90.9% mean_width=2.583 n=176

=== Computing LOO conformal intervals for C2i ===
  α=0.05 target=95% empirical=95.5% mean_width=3.601 n=176
  α=0.10 target=90% empirical=90.3% mean_width=3.372 n=176

=== Computing LOO conformal intervals for C3 ===
  α=0.05 target=95% empirical=95.5% mean_width=3.123 n=176
  α=0.10 target=90% empirical=90.3% mean_width=2.318 n=176

=== Computing LOO conformal intervals for C6 ===
  α=0.05 target=95% empirical=94.9% mean_width=2.667 n=176
  α=0.10 target=90% empirical=90.3% mean_width=1.935 n=176

=== Computing LOO conformal intervals for C7 ===
  α=0.05 target=95% empirical=100.0% mean_width=3.000 n=116
  α=0.10 target=90% empirical=100.0% mean_width=3

In [44]:
# Spec D — empirical coverage diagnostic (per Q-NEW-D.4 full breakdown)
import sqlite3

with sqlite3.connect(DB_PATH) as conn:
    coverage_overall = pd.read_sql(
        """
        SELECT config_name, alpha,
               AVG(in_interval) as empirical_coverage,
               AVG(interval_width) as mean_width,
               COUNT(*) as n
        FROM looo_prediction_intervals
        GROUP BY config_name, alpha
        ORDER BY alpha, empirical_coverage DESC
        """,
        conn
    )

print("=== Empirical coverage per (config × α) ===")
print(coverage_overall.to_string(index=False))

# Per-stage breakdown
with sqlite3.connect(DB_PATH) as conn:
    coverage_by_stage = pd.read_sql(
        """
        SELECT config_name, alpha, stage_code,
               AVG(in_interval) as empirical_coverage,
               AVG(interval_width) as mean_width,
               COUNT(*) as n
        FROM looo_prediction_intervals
        GROUP BY config_name, alpha, stage_code
        ORDER BY config_name, alpha, stage_code
        """,
        conn
    )

print("\n=== Per-stage coverage breakdown ===")
print(coverage_by_stage.to_string(index=False))

# Per-oil_type breakdown (for crude vs refined comparison)
with sqlite3.connect(DB_PATH) as conn:
    coverage_by_oil_type = pd.read_sql(
        """
        SELECT lpi.config_name, lpi.alpha,
               COALESCE(o.oil_type, 'unknown') as oil_type,
               AVG(lpi.in_interval) as empirical_coverage,
               COUNT(*) as n
        FROM looo_prediction_intervals lpi
        LEFT JOIN oils o ON lpi.oil_id = o.oil_id
        GROUP BY lpi.config_name, lpi.alpha, oil_type
        ORDER BY lpi.config_name, lpi.alpha, oil_type
        """,
        conn
    )

print("\n=== Per-oil_type coverage breakdown ===")
print(coverage_by_oil_type.to_string(index=False))


=== Empirical coverage per (config × α) ===
config_name  alpha  empirical_coverage  mean_width   n
         C7   0.05            1.000000    3.000000 116
   C7_mixed   0.05            1.000000    3.000000 176
         C8   0.05            0.956897    3.571637 116
         C1   0.05            0.954545    3.050150 176
         C2   0.05            0.954545    3.078347 176
        C2i   0.05            0.954545    3.601273 176
         C3   0.05            0.954545    3.122797 176
         C6   0.05            0.948864    2.667178 176
      Ridge   0.05            0.948864   11.610666 176
         C7   0.10            1.000000    3.000000 116
   C7_mixed   0.10            1.000000    3.000000 176
         C2   0.10            0.909091    2.582930 176
         C8   0.10            0.905172    2.523513 116
         C1   0.10            0.903409    2.430339 176
        C2i   0.10            0.903409    3.371734 176
         C3   0.10            0.903409    2.318480 176
         C6   0.10   

## Spec E — Wilcoxon Pairwise Distinguishability Test (C1/C3/C6)

Tests whether the aggregate ΔMAE ≤ 0.01 separation between Cohort 1 top-3 (C1, C3, C6) reflects reproducible signal or per-oil noise. Resolves `D-COHORT1-TREE-CLUSTER-09MAI`. Three Wilcoxon signed-rank tests (lexicographic pairs) with Bonferroni correction (family α=0.05), augmented by sign-test sensitivity check, bootstrap 95% CI of median difference, and Wilcoxon MDE simulation. Pre-registered prediction: cluster confirmed (all FTR; signal/noise below detectable band at n=44).


In [ ]:
# Spec E — Wilcoxon pairwise distinguishability test (C1/C3/C6)
# Resolves D-COHORT1-TREE-CLUSTER-09MAI: is aggregate ΔMAE ≤ 0.01 signal or noise?
#
# [PRE-REG, Sessão R, 16/mai/2026]
# Hypothesis: C1/C3/C6 per-oil MAE distributions are statistically
#   indistinguishable at family α=0.05 (Bonferroni × 3).
# Prediction (strong): all 3 Wilcoxon tests FTR (p_corrected ≥ 0.05).
# Weakest pair: C1-C6 (sign 17/27 favoring C1; median_diff=-0.038,
#   sd_diff=0.111; binomial p≈0.18 two-sided).
# Refutation criterion: ≥1 p_corrected < 0.05 → tree-cluster claim
#   falsified; per-oil ranking among C1/C3/C6 is real.
# Mechanism if FTR: signal/noise ≈ 0.09 (mean_diff / sd_diff),
#   below detectable threshold at n=44.
#
# Pair ordering: lexicographic (C1,C3), (C1,C6), (C3,C6) — never reversed.
# Diff convention: config_a - config_b. Negative => config_a better (lower MAE).
# Effect size: Cohen's d_z = mean_diff/sd_diff. MDE in same standardized units.

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, binomtest
from datetime import datetime

ALPHA_FAMILY = 0.05
N_COMPARISONS = 3
ALPHA_CORR = ALPHA_FAMILY / N_COMPARISONS   # 0.01667
BOOTSTRAP_N = 10000
BOOTSTRAP_SEED = 42
MDE_N_SIMS = 5000
MDE_TARGET_POWER = 0.80
MDE_SEED = 42
RUN_ID = f"spec_e_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

PAIRS = [('C1','C3'), ('C1','C6'), ('C3','C6')]   # lexicographic, never reversed

# === Pull per-oil MAE per config ===
with get_conn() as conn:
    df = pd.read_sql("""
        SELECT config_name, oil_id, AVG(abs_error) AS mae_oil
        FROM looo_predictions
        WHERE config_name IN ('C1', 'C3', 'C6')
        GROUP BY config_name, oil_id
    """, conn)
mae_wide = df.pivot(index='oil_id', columns='config_name', values='mae_oil')
assert len(mae_wide) == 44, f'expected 44 oils, got {len(mae_wide)}'
assert mae_wide.isna().sum().sum() == 0, 'unexpected NaN in MAE matrix'
print(f'OK per-oil MAE pulled: {len(mae_wide)} oils x {len(mae_wide.columns)} configs (no NaN)')

# === MDE simulation (Wilcoxon, n=44, alpha_corr=0.0167, power=0.80) ===
def simulate_wilcoxon_power(d_z, n=44, alpha=ALPHA_CORR, n_sims=MDE_N_SIMS, seed=MDE_SEED):
    rng = np.random.default_rng(seed)
    rejects = 0
    for _ in range(n_sims):
        diffs = rng.normal(d_z, 1.0, n)
        _, p = wilcoxon(diffs, alternative='two-sided', zero_method='wilcox')
        if p < alpha:
            rejects += 1
    return rejects / n_sims

print()
print('=== MDE simulation (Wilcoxon, n=44, alpha_corr=0.0167, target power=0.80) ===')
r_grid = np.linspace(0.10, 0.70, 13)
powers = {}
for r in r_grid:
    p = simulate_wilcoxon_power(r)
    powers[r] = p
    print(f'  d_z={r:.3f}  power={p:.3f}')

# Linear-interp MDE between brackets
mde_r = None
sorted_r = sorted(powers.keys())
for r in sorted_r:
    if powers[r] >= MDE_TARGET_POWER:
        below = [x for x in sorted_r if x < r]
        if below:
            r_below = max(below)
            p_below = powers[r_below]
            p_above = powers[r]
            if p_above > p_below:
                frac = (MDE_TARGET_POWER - p_below) / (p_above - p_below)
                mde_r = r_below + frac * (r - r_below)
            else:
                mde_r = r
        else:
            mde_r = r
        break
if mde_r is None:
    mde_r = max(sorted_r)
print()
print(f'  -> MDE (Wilcoxon, n=44, alpha=0.0167, power=0.80): d_z ~ {mde_r:.3f}')

# === Per-pair tests + bootstrap ===
results = []
rng_boot = np.random.default_rng(BOOTSTRAP_SEED)

for ca, cb in PAIRS:
    a = mae_wide[ca].values
    b = mae_wide[cb].values
    diffs = a - b   # config_a - config_b; negative => config_a better

    n = len(diffs)
    mean_diff = float(np.mean(diffs))
    median_diff = float(np.median(diffs))
    sd_diff = float(np.std(diffs, ddof=1))
    sign_pos = int(np.sum(diffs > 0))
    sign_neg = int(np.sum(diffs < 0))
    sign_zero = int(np.sum(diffs == 0))
    d_z = mean_diff / sd_diff if sd_diff > 0 else 0.0

    # Wilcoxon
    w_stat, w_p = wilcoxon(diffs, alternative='two-sided', zero_method='wilcox')
    w_p_corr = min(float(w_p) * N_COMPARISONS, 1.0)

    # Sign test
    n_nonzero = sign_pos + sign_neg
    if n_nonzero > 0:
        s_p = float(binomtest(sign_pos, n_nonzero, p=0.5, alternative='two-sided').pvalue)
        s_stat = sign_pos
    else:
        s_p = 1.0
        s_stat = 0
    s_p_corr = min(s_p * N_COMPARISONS, 1.0)

    # Bootstrap 95% CI of median diff
    boot_medians = np.empty(BOOTSTRAP_N)
    for i in range(BOOTSTRAP_N):
        idx = rng_boot.integers(0, n, size=n)
        boot_medians[i] = np.median(diffs[idx])
    ci_lo = float(np.percentile(boot_medians, 2.5))
    ci_hi = float(np.percentile(boot_medians, 97.5))

    common = dict(
        config_a=ca, config_b=cb, n_pairs=n,
        mean_diff=mean_diff, median_diff=median_diff, sd_diff=sd_diff,
        ci_lo=ci_lo, ci_hi=ci_hi,
        sign_pos=sign_pos, sign_neg=sign_neg, sign_zero=sign_zero,
    )
    results.append({**common, 'test_name': 'wilcoxon_signed_rank',
                    'statistic': float(w_stat), 'p_value': float(w_p), 'p_corrected': w_p_corr,
                    'effect_size': d_z, 'min_detectable_r': float(mde_r)})
    results.append({**common, 'test_name': 'sign_test',
                    'statistic': float(s_stat), 'p_value': s_p, 'p_corrected': s_p_corr,
                    'effect_size': None, 'min_detectable_r': None})

# === Persist ===
with get_conn() as conn:
    for r in results:
        conn.execute("""
            INSERT OR REPLACE INTO looo_pairwise_tests
            (config_a, config_b, n_pairs, test_name, statistic, p_value, p_corrected,
             alpha_family, n_comparisons, effect_size, mean_diff, median_diff, sd_diff,
             ci_lo, ci_hi, sign_pos, sign_neg, sign_zero, min_detectable_r, run_id)
            VALUES (:config_a, :config_b, :n_pairs, :test_name, :statistic, :p_value, :p_corrected,
                    :alpha_family, :n_comparisons, :effect_size, :mean_diff, :median_diff, :sd_diff,
                    :ci_lo, :ci_hi, :sign_pos, :sign_neg, :sign_zero, :min_detectable_r, :run_id)
        """, {**r, 'alpha_family': ALPHA_FAMILY, 'n_comparisons': N_COMPARISONS, 'run_id': RUN_ID})
    conn.commit()
print()
print(f'OK persisted {len(results)} rows to looo_pairwise_tests (run_id={RUN_ID})')

# === Summary ===
print()
print('=== SPEC E RESULTS ===')
print(f'{"Pair":<8} {"Test":<22} {"Stat":>10} {"p":>10} {"p_corr":>10} {"d_z":>8} {"Verdict":<24}')
print('-' * 95)
for r in results:
    pair = f'{r["config_a"]}-{r["config_b"]}'
    verdict = 'distinguishable' if r['p_corrected'] < ALPHA_FAMILY else 'NOT distinguishable'
    dz_str = f'{r["effect_size"]:+.3f}' if r['effect_size'] is not None else 'n/a'
    print(f'{pair:<8} {r["test_name"]:<22} {r["statistic"]:>10.4f} {r["p_value"]:>10.4f} {r["p_corrected"]:>10.4f} {dz_str:>8} {verdict:<24}')

print()
print(f'  MDE (Wilcoxon, n=44, alpha_corr=0.0167, power=0.80): d_z ~ {mde_r:.3f}')
print('  Observed |d_z| per pair vs MDE:')
for r in results:
    if r['test_name'] == 'wilcoxon_signed_rank':
        ratio = abs(r['effect_size']) / mde_r if mde_r > 0 else 0
        print(f'    {r["config_a"]}-{r["config_b"]}: |d_z|={abs(r["effect_size"]):.3f}  |observed|/MDE = {ratio:.2f}')

# Family verdict
w_rejects = [r for r in results if r['test_name'] == 'wilcoxon_signed_rank' and r['p_corrected'] < ALPHA_FAMILY]
s_rejects = [r for r in results if r['test_name'] == 'sign_test' and r['p_corrected'] < ALPHA_FAMILY]
print()
if not w_rejects and not s_rejects:
    print('  -> Family verdict: C1/C3/C6 statistically indistinguishable at alpha_family=0.05')
    print('  -> D-SPEC-E-CLUSTER-CONFIRMED-16MAI')
    print(f'  -> MDE-bounded null: operationally equivalent within d_z < {mde_r:.2f} (n=44)')
elif w_rejects:
    print(f'  -> Family verdict: {len(w_rejects)}/3 Wilcoxon pair(s) distinguishable')
    print('  -> D-SPEC-E-PARTIAL-DIFFERENTIATION-16MAI')
    if not s_rejects:
        print('  ! Sensitivity flag: Wilcoxon rejects but sign test does not - review symmetry')
else:
    print('  ! Sensitivity flag: sign test rejects but Wilcoxon does not - review')